In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gona26/tinystories-data-540m/dataset_stats.txt
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00002.tok.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00000.tok.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00003.doc.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00005.doc.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00001.tok.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00001.doc.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00002.doc.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00004.doc.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00003.tok.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00005.tok.npy
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val/shard_00004.tok.npy
/kaggle/input/dat

In [2]:
%%writefile data_pipeline.py

"""
data_pipeline.py -- PIPELINE A: raw documents -> token shards (run ONCE).

Responsibilities:
  SETUP    Tokenizer, Config (shared vocab/model sizes)
  STAGE 0  RawCorpus   -- load raw text documents from a folder
  STAGE 1  ShardWriter -- tokenize docs -> token shards saved to disk
  STAGE 2  ShardStream -- load a token budget of shards into RAM -> batches

Run once (offline):
    python data_pipeline.py

Tokens are pre-tokenized and SAVED here, never tokenized on the fly.
learn_pipeline.py starts from the shards this produces.
"""

import os
import glob

import numpy as np
from tokenizers import ByteLevelBPETokenizer


# where the trained BPE lives, what it trains on, and the target vocab size
BPE_DIR = "data/bpe"
RAW_DIR = "data/raw"
VOCAB_SIZE = 4000        # target; actual may be smaller on a tiny corpus


# ---- SETUP -----------------------------------------------------------------
class Tokenizer:
    """Byte-level BPE trained on YOUR corpus (small vocab -> small embeddings).

    First construction trains the BPE from RAW_DIR and saves it to BPE_DIR;
    later constructions just load it. PAD/EOS are reserved ids just past the
    learned vocab (so the model sizes its embedding to V = vocab + 2).
    """

    def __init__(self, bpe_dir=BPE_DIR, raw_dir=RAW_DIR, vocab_size=VOCAB_SIZE):
        vocab_json = os.path.join(bpe_dir, "vocab.json")
        merges_txt = os.path.join(bpe_dir, "merges.txt")
        if not (os.path.exists(vocab_json) and os.path.exists(merges_txt)):
            self._train(bpe_dir, raw_dir, vocab_size)
        self.bpe = ByteLevelBPETokenizer(vocab_json, merges_txt)
        n = self.bpe.get_vocab_size()
        self.PAD = n          # reserved ids just past the learned vocab
        self.EOS = n + 1
        self.V = n + 2        # total vocab the model must size its embedding to

    def _train(self, bpe_dir, raw_dir, vocab_size):
        files = sorted(glob.glob(os.path.join(raw_dir, "*.txt")))
        if not files:
            raise SystemExit(f"no .txt in {raw_dir}/ to train the BPE on")
        total_mb = sum(os.path.getsize(f) for f in files) / 1e6
        print(f"[BPE] training on {len(files)} files ({total_mb:.1f} MB), "
              f"target vocab {vocab_size} ...", flush=True)
        os.makedirs(bpe_dir, exist_ok=True)
        bpe = ByteLevelBPETokenizer()
        # min_frequency=1 lets it keep merging until it HITS vocab_size exactly
        bpe.train(files=files, vocab_size=vocab_size, min_frequency=1)
        bpe.save_model(bpe_dir)
        got = bpe.get_vocab_size()
        print(f"[BPE] done: {got} tokens (target {vocab_size}) -> {bpe_dir}/", flush=True)
        if got < vocab_size:
            print(f"[BPE] WARNING: only {got} < {vocab_size} -- corpus too small", flush=True)

    def encode(self, text):
        return self.bpe.encode(text).ids

    def decode(self, ids):
        ids = [i for i in ids if i not in (self.PAD, self.EOS)]
        return self.bpe.decode(ids)

    def piece(self, i):
        if i == self.PAD: return "<pad>"
        if i == self.EOS: return "<eos>"
        return self.bpe.decode([i])


class Config:
    def __init__(self):
        self.TOK = Tokenizer()
        self.V = self.TOK.V  # vocabulary size (from the BPE tokenizer)
        self.T = 256     # context window (fits a whole TinyStory ~200 tok)
        self.D = 256     # model dimension
        self.H = 8      # number of attention heads
        self.HD = self.D // self.H  # head dimension
        self.DFF = self.D*4  # feed-forward dimension
        self.L = 6      # number of layers


# ---- STAGE 0: load raw documents ------------------------------------------
class RawCorpus:
    """A folder of raw text files; each file is one document.

    Drop your own .txt files into `folder` (one document per file) and the
    rest of the pipeline picks them up. docs() streams files lazily so the
    whole corpus never needs to sit in RAM.
    """

    def __init__(self, folder, pattern="*.txt"):
        self.folder = folder
        self.pattern = pattern

    def paths(self):
        return sorted(glob.glob(os.path.join(self.folder, self.pattern)))

    def docs(self):
        for p in self.paths():
            with open(p, "r", encoding="utf-8") as f:
                yield f.read()

    def __len__(self):
        return len(self.paths())


# ---- STAGE 1: tokenize -> token shards on disk (run once) -----------------
class ShardWriter:
    """Tokenize a corpus -> fixed-size train/val shards.

    Each shard i -> two parallel files under out_dir/<split>/:
        shard_{i}.tok.npy  token ids (uint16)
        shard_{i}.doc.npy  doc ids   (uint32)  -> lets the trainer rebuild the
                                                   cross-document attention mask.
    Real pipelines use ~100M tokens/shard; we keep it tiny to see the mechanics.
    """

    TOKEN_DTYPE = np.uint16
    DOCID_DTYPE = np.uint32

    def __init__(self, tokenizer, out_dir="data/shards", shard_size=1000,
                 val_frac=0.1):
        self.tok = tokenizer
        self.out_dir = out_dir
        self.shard_size = shard_size
        self.val_frac = val_frac

    def _write_split(self, docs, split):
        split_dir = os.path.join(self.out_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        tok_buf, doc_buf, idx = [], [], 0

        def flush(tok_buf, doc_buf, idx):
            base = os.path.join(split_dir, f"shard_{idx:05d}")
            np.save(base + ".tok.npy", np.array(tok_buf, dtype=self.TOKEN_DTYPE))
            np.save(base + ".doc.npy", np.array(doc_buf, dtype=self.DOCID_DTYPE))
            print(f"  {split}/shard_{idx:05d}  {len(tok_buf):>6} tokens")

        for d, text in enumerate(docs):
            ids = self.tok.encode(text) + [self.tok.EOS]   # EOS separates docs
            tok_buf.extend(ids)
            doc_buf.extend([d] * len(ids))
            while len(tok_buf) >= self.shard_size:         # emit full shards
                flush(tok_buf[:self.shard_size], doc_buf[:self.shard_size], idx)
                tok_buf = tok_buf[self.shard_size:]
                doc_buf = doc_buf[self.shard_size:]
                idx += 1
        if tok_buf:                                        # final partial shard
            flush(tok_buf, doc_buf, idx)
        return sorted(glob.glob(os.path.join(split_dir, "shard_*.tok.npy")))

    def build(self, corpus):
        docs = list(corpus.docs())
        n_val = max(1, int(len(docs) * self.val_frac))
        val_docs, train_docs = docs[:n_val], docs[n_val:]
        print(f"docs: {len(train_docs)} train / {len(val_docs)} val")
        print("writing train shards:")
        train_paths = self._write_split(train_docs, "train")
        print("writing val shards:")
        val_paths = self._write_split(val_docs, "val")
        print(f"done: {len(train_paths)} train + {len(val_paths)} val shards")
        return train_paths, val_paths


# ---- STAGE 2: load a token budget of shards into RAM -> batches -----------
class ShardStream:
    """Stream (tokens, doc_id) (B, T) batches from a split's shards.

    - Loads only as many shards as needed to reach max_tokens (None = all),
      straight into RAM ONCE. 50M tokens = ~100 MB tokens + ~200 MB doc ids.
      After that, every batch is pure in-memory indexing: no file opens.
    - Non-overlapping blocks: the loaded tokens are cut into back-to-back
      T-length blocks so no token window is ever reused.
    - Rank partition: all ranks share one seed -> identical global block shuffle,
      then rank r takes blocks[r::world_size] -> disjoint data, no overlap.
    - Defaults rank=0/world_size=1 -> single process gets ALL blocks.
    - Yields numpy int64 arrays; the trainer wraps them in torch tensors.
    """

    def __init__(self, split_dir, B, T, seed=0, max_tokens=None, max_shards=None,
                 rank=0, world_size=1):
        paths = sorted(glob.glob(os.path.join(split_dir, "shard_*.tok.npy")))
        if max_shards is not None:                 # None = all shards; int = first N
            paths = paths[:max_shards]
        if not paths:
            raise SystemExit(f"no shards in {split_dir}/")

        toks, docs, have = [], [], 0
        for p in paths:                            # stop once the budget is met
            if max_tokens is not None and have >= max_tokens:
                break
            toks.append(np.load(p))
            docs.append(np.load(p.replace(".tok.npy", ".doc.npy")))
            have += len(toks[-1])
        self.toks = np.concatenate(toks)
        self.docs = np.concatenate(docs)
        if max_tokens is not None:                 # trim to the exact budget
            self.toks = self.toks[:max_tokens]
            self.docs = self.docs[:max_tokens]

        self.B, self.T = B, T
        self.rank, self.world_size = rank, world_size
        self.rng = np.random.default_rng(seed)     # SHARED across ranks (same seed)
        self.n_blocks = len(self.toks) // T        # global block id runs 0..n_blocks-1
        self.offsets = np.arange(T)
        print(f"{split_dir}: {len(toks)} shards, {len(self.toks):,} tokens, "
              f"{self.n_blocks:,} blocks", flush=True)

    def _batch(self, block_ids):
        idx = block_ids[:, None] * self.T + self.offsets   # (B, T) positions
        return self.toks[idx].astype(np.int64), self.docs[idx].astype(np.int64)

    def epoch(self):
        order = self.rng.permutation(self.n_blocks)      # same on all ranks (shared seed)
        mine = order[self.rank::self.world_size]          # disjoint stride partition
        for i in range(0, len(mine) - self.B + 1, self.B):
            yield self._batch(mine[i:i + self.B])         # one vectorized gather

    def iter_batches(self):
        """Infinite stream: loop epochs forever (for a fixed step budget)."""
        while True:
            yield from self.epoch()


def preprocess_to_shards(cfg, raw_dir="data/raw", shard_dir="data/shards",
                         shard_size=1000, val_frac=0.1, overwrite=False):
    """One-time preprocess: read the raw folder and write token shards.

    Skips if shards already exist (pass overwrite=True to force a rebuild).
    """
    have = glob.glob(os.path.join(shard_dir, "train", "shard_*.tok.npy"))
    if have and not overwrite:
        print(f"shards already in {shard_dir}/ -- skip (overwrite=True to rebuild)")
        return
    corpus = RawCorpus(raw_dir)
    if len(corpus) == 0:
        raise SystemExit(f"no .txt files in {raw_dir}/ -- drop documents there first")
    print(f"loaded {len(corpus)} docs from {raw_dir}/")
    ShardWriter(cfg.TOK, out_dir=shard_dir, shard_size=shard_size,
                val_frac=val_frac).build(corpus)


def preprocess_split(cfg, raw_dir, split, shard_dir="data/shards", shard_size=1_000_000):
    """Tokenize ALL docs in raw_dir into shards under shard_dir/<split>/.

    Unlike preprocess_to_shards (which splits ONE folder by val_frac), this writes
    a given folder to a named split -- use it to build train and val from the two
    separate TinyStories files (no leakage, real held-out val).
    """
    corpus = RawCorpus(raw_dir)
    if len(corpus) == 0:
        raise SystemExit(f"no .txt files in {raw_dir}/")
    writer = ShardWriter(cfg.TOK, out_dir=shard_dir, shard_size=shard_size)
    paths = writer._write_split(list(corpus.docs()), split)
    print(f"{split}: {len(paths)} shards from {raw_dir}/")
    return paths


def get_streams(cfg, shard_dir="data/shards", micro_B=4, seed=0,
                train_tokens=50_000_000, val_tokens=None, max_shards=None,
                rank=0, world_size=1):
    """Load token budgets into RAM; return (train_stream, val_stream).

    train_tokens: tokens to load for training (default 50M). None = all shards.
    val_tokens:   tokens to load for validation. None = all val shards.
    max_shards:   optional extra cap on shard count (e.g. 1 for quick debugging).
    rank/world_size: pass the SAME seed on every rank; each rank gets a disjoint
                     slice of blocks. Defaults (0, 1) = single process, all data.
    train_stream: infinite micro-batch iterator for the training loop.
    val_stream:   a callable returning one finite pass over val.
    """
    train = ShardStream(os.path.join(shard_dir, "train"), B=micro_B, T=cfg.T,
                        seed=seed, max_tokens=train_tokens, max_shards=max_shards,
                        rank=rank, world_size=world_size)
    val = ShardStream(os.path.join(shard_dir, "val"), B=micro_B, T=cfg.T,
                      seed=seed + 123, max_tokens=val_tokens, max_shards=max_shards,
                      rank=rank, world_size=world_size)
    return train.iter_batches(), val.epoch


if __name__ == "__main__":
    # Run once (offline): train BPE on data/raw, then raw docs -> token shards.
    # overwrite=True so old (gpt2-tokenized) shards are rebuilt with the new vocab.
    cfg = Config()
    print(f"vocab V = {cfg.V}  (BPE {cfg.V - 2} + PAD + EOS)")
    preprocess_to_shards(cfg, overwrite=True)

Writing data_pipeline.py


In [3]:
%%writefile learn_pipeline.py

"""
learn_pipeline.py -- PIPELINE B: token shards -> batches -> probabilities.

All DATA handling (tokenizer, shard writing, shard streaming) lives in
data_pipeline.py. This file starts from the streamed batches and turns them
into logits -> probabilities -> loss (the learning half).

    python data_pipeline.py     # once: raw -> token shards
    python learn_pipeline.py    # every run: shards -> batches (-> training)
"""

import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from data_pipeline import Config, get_streams, Tokenizer

# build the tokenizer ONCE (module-level); reloading it per make_masks call was
# reading the BPE files off disk every step -> a real per-step slowdown
_TOK = None

def make_masks(tokens,doc_id):
    global _TOK
    if _TOK is None:
        _TOK = Tokenizer()          # first call only; cached thereafter
    TOK = _TOK
    B,T = tokens.shape
    # no peeking into future tokens
    casual = torch.tril(torch.ones(T,T, dtype=torch.bool, device=tokens.device)) # (T,T)
    # only attend to tokens within the same document
    same = doc_id[:,:,None] == doc_id[:,None,:] # (B,T,1) vs (B,1,T) -> (B,T,T)
    # combine causal and same-document masks
    attn = (casual & same).unsqueeze(1) # (B,1,T,T)
    # target tokens (shifted by one)
    tgt = tokens[:,1:]
    # identify positions where the document changes
    join = doc_id[:,1:] != doc_id[:,:-1] # (B,T-1) vs (B,T-1) -> comparing adjacent doc_ids
    # positions where the document changes (to avoid computing loss across document boundaries)
    loss = (tgt != TOK.PAD) & (~join)
    return attn, loss

class RMSNorm(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.d = d
        self.g = nn.Parameter(torch.ones(d))
    def forward(self,x):
        # RMSNorm forward pass
        ms = x.pow(2).mean(-1, keepdim=True) # (B,T,D) -> (B,T,1)
        return x*torch.rsqrt(ms + 1e-6)*self.g

class Attention(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.c = c
        self.q = nn.Linear(c.D, c.D, bias=False)
        self.k = nn.Linear(c.D, c.D, bias=False)
        self.v = nn.Linear(c.D, c.D, bias=False)
        self.out = nn.Linear(c.D, c.D, bias=False)
    def forward(self,x,mask):
        c  = self.c
        B,T,_ = x.shape
        sp = lambda t: t.view(B,T,c.H, c.HD).transpose(1,2)
        # get q,k,v projections for multi-head attention
        q,k,v = sp(self.q(x)), sp(self.k(x)), sp(self.v(x)) # (B,c.H,T,c.HD)
        # multiply queries and keys to get attention scores
        s = (q @ k.transpose(-2,-1)) / (c.HD ** 0.5) # (B,c.H,T,c.HD) @ (B,c.H,c.HD,T) -> (B,c.H,T,T)
        # apply the attention mask to prevent attending to future tokens or tokens from different documents
        s = s.masked_fill(~mask, float('-inf'))
        # softmax to obtain attention weights
        w = torch.nan_to_num(s.softmax(-1))
        # value vectors are combined with attention weights to produce the context vectors
        ctx = (w @ v).transpose(1,2).reshape(B,T,c.D) 
        return self.out(ctx)

class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.D, c.DFF, bias=False)
        self.up = nn.Linear(c.D, c.DFF, bias=False)
        self.bottom = nn.Linear(c.DFF, c.D, bias=False)
    def forward(self,x):
        return self.bottom(self.up(x) * F.silu(self.gate(x)))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.attn = Attention(c)
        self.n1 = RMSNorm(c.D)
        self.n2 = RMSNorm(c.D)
        self.ffn = SwiGLU(c)
    def forward(self,x,mask):
        x = x + self.attn(self.n1(x), mask)
        x = x + self.ffn(self.n2(x))
        return x


class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok_emb = nn.Embedding(c.V, c.D)
        self.pos_emb = nn.Embedding(c.T, c.D)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.L)])
        self.ns = RMSNorm(c.D)
        self.head = nn.Linear(c.D, c.V, bias=False)

    def forward(self,tokens, mask):
        B,T = tokens.shape
        pos = torch.arange(T, device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x,mask)
        x = self.ns(x)
        logits = self.head(x)
        return logits

def loss_fn(logits, tokens, loss_mask):
    tgt = tokens[:, 1:].reshape(-1) # (B, T-1) -> (B*(T-1))
    lg = logits[:,:-1].reshape(-1,logits.size(-1)) # (B, T-1, V) -> (B*(T-1), V)
    nll = F.cross_entropy(lg, tgt, reduction='none')
    m = loss_mask.reshape(-1).float()
    loss = (nll * m).sum() / m.sum().clamp(min=1)
    return loss, int(m.sum())

def lr_at(s, WARMUP, STEPS):
    if s < WARMUP:
        return s / WARMUP
    t = (s - WARMUP) / max(1, STEPS - WARMUP)
    return 0.1 + 0.45 * (1 + math.cos(math.pi * t))


# @torch.no_grad()
# def evaluate(model, val_stream, max_batches=20):
#     """Token-weighted mean loss over up to max_batches of the val stream."""
#     model.eval()
#     tot, ntok = 0.0, 0
#     for i, (x, d) in enumerate(val_stream()):        # fresh finite pass
#         if i >= max_batches:
#             break
#         tokens, doc_id = torch.from_numpy(x), torch.from_numpy(d)
#         attn, loss_mask = make_masks(tokens, doc_id)
#         loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
#         tot += loss.item() * n
#         ntok += n
#     model.train()
#     return tot / max(1, ntok)


# def _trace_shapes(model, tokens, doc_id):
#     """One-time forward pass that prints the output shape of every submodule."""
#     attn, loss_mask = make_masks(tokens, doc_id)
#     print("  inputs:")
#     print(f"    {'tokens':<26} {tuple(tokens.shape)}")
#     print(f"    {'doc_id':<26} {tuple(doc_id.shape)}")
#     print(f"    {'attn mask':<26} {tuple(attn.shape)}")
#     print(f"    {'loss mask':<26} {tuple(loss_mask.shape)}")
#     print("  module outputs:")

#     handles = []
#     def mk(name):
#         def hook(mod, inp, out):
#             if isinstance(out, torch.Tensor):
#                 print(f"    {name:<26} {tuple(out.shape)}")
#         return hook
#     for name, mod in model.named_modules():
#         if name:                                     # skip the root module
#             handles.append(mod.register_forward_hook(mk(name)))
#     with torch.no_grad():
#         logits = model(tokens, attn)
#     for h in handles:
#         h.remove()
#     print(f"    {'logits (head)':<26} {tuple(logits.shape)}")


# def train(STEPS=500, WARMUP=50, lr=3e-4, micro_B=4, accum_steps=4,
#           eval_every=100, log_every=1, max_shards=None, seed=0, debug=True):
#     cfg = Config()
#     torch.manual_seed(seed)

#     # STAGE 2: stream (tokens, doc_id) batches from the saved token shards.
#     train_stream, val_stream = get_streams(cfg, micro_B=micro_B, seed=seed,
#                                            max_shards=max_shards)
#     model = GPT(cfg)
#     opt = torch.optim.AdamW(model.parameters(), lr=lr)
#     print(f"eff. batch {micro_B * accum_steps}  |  ln(V) {math.log(cfg.V):.3f}")

#     if debug:                                        # one-time shape trace
#         x, d = next(train_stream)
#         print("shape trace:")
#         _trace_shapes(model, torch.from_numpy(x), torch.from_numpy(d))

#     seen = 0                                         # cumulative tokens trained on
#     for s in range(STEPS):
#         t0 = time.time()
#         cur_lr = lr * lr_at(s, WARMUP, STEPS)
#         for g in opt.param_groups:                   # LR warmup + cosine decay
#             g['lr'] = cur_lr
#         opt.zero_grad(set_to_none=True)

#         run_loss, step_tok = 0.0, 0
#         for _ in range(accum_steps):                 # accumulate micro-batches
#             x, d = next(train_stream)
#             tokens, doc_id = torch.from_numpy(x), torch.from_numpy(d)
#             attn, loss_mask = make_masks(tokens, doc_id)
#             loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
#             (loss / accum_steps).backward()          # grads add up; no step yet
#             run_loss, step_tok = run_loss + loss.item() * n, step_tok + n
#         gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         opt.step()                                   # one update per effective batch
#         seen += step_tok

#         if s % log_every == 0:
#             dt = time.time() - t0
#             avg = run_loss / max(1, step_tok)        # token-weighted mean over the step
#             print(f"step {s:4d} | lr {cur_lr:.2e} | loss {avg:.4f} | "
#                   f"gnorm {float(gnorm):.2f} | tok {step_tok} | seen {seen} | "
#                   f"{step_tok / max(dt, 1e-9):.0f} tok/s")
#         if s % eval_every == 0:
#             print(f"           val {evaluate(model, val_stream):.4f}")
#     return model


if __name__ == "__main__":
    # Requires token shards on disk first:  python data_pipeline.py
    # train(STEPS=500, eval_every=50)
    pass



Writing learn_pipeline.py


In [4]:
%%writefile zero3_launch.py

# read env vars torchrun injects
import os
# csv metrics logging + step timing
import csv
import time
import math
# add the Kaggle dataset folder to the import path
import sys
# for torch.cuda.set_device
import torch
# collectives: init/all_reduce/broadcast
import torch.distributed as dist
# ZeRO-3 engine: shards params + grads + optimizer, gathers layer-by-layer
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
from torch.distributed.fsdp.wrap import ModuleWrapPolicy

# dataset root holds the loose .py files; add it to the import path
SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m"
# shards + bpe live under this subfolder
DATA = f"{SRC}/"
sys.path.insert(0, SRC)
# model/vocab sizes + the data stream factory
from data_pipeline import Config, get_streams
# model + a Block (the unit we shard) + RMSNorm (needs custom init) + helpers
from learn_pipeline import GPT, Block, RMSNorm, make_masks, loss_fn, lr_at


def setup():
    # CPU: read this process's GPU index that torchrun injected (str -> int)
    local_rank = int(os.environ["LOCAL_RANK"])
    # GPU: pin this process to that one GPU (must happen before init)
    torch.cuda.set_device(local_rank)
    # GPU<->GPU: join the process group; device_id tells NCCL our GPU
    dist.init_process_group(
        backend="nccl",
        device_id=torch.device(f"cuda:{local_rank}"),
    )
    # CPU: return identity read back from the group
    return dist.get_rank(), dist.get_world_size(), local_rank


def print0(rank, *args):
    if rank == 0:
        print(*args, flush=True)


def model_info(model, cfg):
    P = sum(p.numel() for p in model.parameters())    # numel works on meta tensors too
    emb = 2 * cfg.V * cfg.D
    return (f"model: {P/1e6:.2f}M params | L={cfg.L} D={cfg.D} H={cfg.H} "
            f"T={cfg.T} V={cfg.V} | embed+head ~{emb/1e6:.1f}M ({100*emb/P:.0f}%)")


def measure_holdings(model, opt):
    # actual bytes from live tensors; under FSDP these are the LOCAL shards (1/N)
    mb = lambda ts: sum(t.numel() * t.element_size() for t in ts) / 1e6
    params_mb = mb(list(model.parameters()))
    grads_mb = mb([p.grad for p in model.parameters() if p.grad is not None])
    optim_mb = mb([t for st in opt.state.values() for t in st.values() if torch.is_tensor(t)])
    return params_mb, grads_mb, optim_mb


def param_init_fn(module):
    # FSDP calls this per module while sharding, to bring META tensors to life:
    # allocate real (uninitialized) tensors on THIS process's GPU...
    module.to_empty(device=torch.cuda.current_device(), recurse=False)
    # ...then set proper initial values (meta tensors carried none)
    with torch.no_grad():
        if isinstance(module, RMSNorm):
            module.g.fill_(1.0)                 # RMSNorm gain starts at 1
        elif hasattr(module, "reset_parameters"):
            module.reset_parameters()           # Linear/Embedding default init


@torch.no_grad()
def evaluate(model, val_stream, device, max_batches):
    """Token-weighted mean loss over up to max_batches of the held-out val set,
    summed across ranks -> a true GLOBAL validation loss (perplexity = e^loss)."""
    model.eval()
    tot = torch.zeros(2, device=device)              # [sum loss*tok, sum tok]
    for i, (x, d) in enumerate(val_stream()):
        if i >= max_batches:
            break
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        attn, loss_mask = make_masks(tokens, doc_id)
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        tot[0] += loss.item() * n
        tot[1] += n
    dist.all_reduce(tot)                             # GPU<->GPU: global sums
    model.train()
    return (tot[0] / tot[1].clamp(min=1)).item()


def main():
    rank, world_size, local_rank = setup()
    device = torch.device(f"cuda:{local_rank}")
    print(f"rank {rank}/{world_size} on cuda:{local_rank}  \n", flush=True)

    # ---- data: disjoint blocks per rank (shared seed) ----
    # CPU: Config builds tokenizer; the stream reads shards into numpy (numpy = CPU RAM)
    cfg = Config()
    # guarantee the ~19.6M ("20M") config no matter which data_pipeline is uploaded
    cfg.D, cfg.H, cfg.L = 384, 8, 7
    cfg.T = 256                                    # longer context (fits a whole story)
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    micro_B = 16                                   # halved for T=256 (attention is O(T^2))
    # CPU: sets up the numpy batch iterators (no GPU touched here); keep val too
    train_stream, val_stream = get_streams(cfg, shard_dir=f"{DATA}/shards",
                                       micro_B=micro_B, seed=1234,
                                       train_tokens=50_000_000,
                                       val_tokens=None,
                                       rank=rank, world_size=world_size)

    # 50M-token budget: STEPS = 50e6 / (T * micro_B * world_size)
    STEPS, WARMUP, base_lr, LOG_EVERY = 6104, 300, 3e-4, 50
    EVAL_EVERY, EVAL_BATCHES = 500, 50            # held-out val perplexity cadence
    torch.manual_seed(0)
    # META init: build the model with SHAPES ONLY -> ZERO bytes allocated, so the full
    # model never lands on any single GPU (this is what avoids OOM for huge models).
    with torch.device("meta"):
        model = GPT(cfg)
    print0(rank, model_info(model, cfg))

    # ---- ZeRO-3: shard params across ranks; FSDP materializes only each rank's 1/N ----
    # GPU: FULL_SHARD = ZeRO-3 (params + grads + optimizer sharded). param_init_fn builds
    # + inits each shard on the GPU, so the whole model is never assembled in one place.
    # (SHARD_GRAD_OP would be ZeRO-2; NO_SHARD would be plain DDP.)
    model = FSDP(
        model,
        # each Block is gathered/freed on its own -> peak memory = ~one Block
        auto_wrap_policy=ModuleWrapPolicy({Block}),
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        device_id=local_rank,
        param_init_fn=param_init_fn,
    )
    # CPU setup over GPU-resident sharded params: MUST be built AFTER FSDP so it sees
    # the sharded (flat) params; its m/v state is allocated on GPU at first step, 1/N size
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr)
    print0(rank, model)

    if rank == 0:
        logf = open("/kaggle/working/metrics_zero3.csv", "w", newline="")
        writer = csv.writer(logf)
        writer.writerow(["step", "tokens", "loss", "ppl", "val_loss", "val_ppl",
                         "grad_norm", "tok_per_s", "lr", "mem_mb"])

    tok_per_step = cfg.T * micro_B * world_size
    for s in range(1, STEPS + 1):
        t0 = time.time()
        # CPU: schedule math (plain floats), then push the LR onto the optimizer
        cur_lr = base_lr * lr_at(s, WARMUP, STEPS)
        for g in opt.param_groups:
            g['lr'] = cur_lr
        # CPU: pull one numpy micro-batch off the stream
        x, d = next(train_stream)
        # CPU->GPU: numpy -> GPU tensors (from_numpy is CPU, .to(device) copies over)
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        # GPU: build the attention/loss masks on the data's device
        attn, loss_mask = make_masks(tokens, doc_id)
        # GPU (+ GPU<->GPU comms): FSDP forward all_gathers each Block's full params
        # just-in-time, runs it, then FREES them -> peak memory ~ one Block
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        # GPU: drop stale grads
        opt.zero_grad(set_to_none=True)
        # GPU (+ comms): backward gathers params again, computes grads, then
        # reduce_scatters them -> each rank KEEPS ONLY ITS SHARD of the gradients
        loss.backward()
        # GPU (+ comms): FSDP-aware global grad-norm clip; returns the total norm
        gnorm = model.clip_grad_norm_(1.0)
        # GPU: each rank updates ONLY its param shard (no manual all_reduce/broadcast)
        opt.step()
        # CPU: wall-clock for this step (GPU work already synced by the collectives)
        dt = time.time() - t0
        seen = s * tok_per_step

        if s % LOG_EVERY == 0:
            # GPU<->GPU: sum (loss*tokens) and tokens across ranks -> true global mean
            stats = torch.tensor([loss.item() * n, float(n)], device=device)
            dist.all_reduce(stats)
            # GPU->CPU: .item() pulls the scalar back to host for printing/logging
            avg_loss = (stats[0] / stats[1]).item()
            ppl = math.exp(min(avg_loss, 20))            # train perplexity = e^loss

            # held-out validation perplexity (the real-world quality signal)
            val_loss = val_ppl = ""
            if s % EVAL_EVERY == 0 or s == STEPS:
                vl = evaluate(model, val_stream, device, EVAL_BATCHES)
                val_loss, val_ppl = f"{vl:.4f}", f"{math.exp(min(vl, 20)):.2f}"

            if rank == 0:
                # GPU->CPU: read peak memory counter
                mem = torch.cuda.max_memory_allocated() / 1e6
                tps = tok_per_step / max(dt, 1e-9)
                eta_min = (STEPS - s) * dt / 60           # rough ETA from this step
                # CPU: write the row to disk
                writer.writerow([s, seen, f"{avg_loss:.4f}", f"{ppl:.2f}",
                                 val_loss, val_ppl, f"{float(gnorm):.2f}",
                                 f"{tps:.0f}", f"{cur_lr:.2e}", f"{mem:.0f}"])
                logf.flush()
                msg = (f"step {s:5d}/{STEPS} | {seen/1e6:5.1f}M tok | "
                       f"loss {avg_loss:.4f} ppl {ppl:.1f}")
                if val_ppl:
                    msg += f" | val_ppl {val_ppl}"
                msg += (f" | gnorm {float(gnorm):.2f} | {tps:.0f} tok/s | "
                        f"mem {mem:.0f}MB | eta {eta_min:.0f}m")
                print(msg, flush=True)

    if rank == 0:
        logf.close()

    # per-rank holdings: params + grads + optim should ALL be ~1/world_size now
    params_mb, grads_mb, optim_mb = measure_holdings(model, opt)
    print(f"rank {rank} holdings: params {params_mb:.0f}MB grads {grads_mb:.0f}MB "
          f"optim {optim_mb:.0f}MB total {params_mb+grads_mb+optim_mb:.0f}MB", flush=True)
    if rank == 0:
        with open("/kaggle/working/holdings_zero3.csv", "w", newline="") as hf:
            w = csv.writer(hf)
            w.writerow(["component", "mb"])
            w.writerow(["params", f"{params_mb:.2f}"])
            w.writerow(["grads", f"{grads_mb:.2f}"])
            w.writerow(["optimizer", f"{optim_mb:.2f}"])

    dist.barrier()
    print0(rank, "done")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

# ---- after training, plot / compare (normal notebook cell) ----
#   from plot import plot_holdings, compare
#   plot_holdings("/kaggle/working/holdings_zero3.csv", "/kaggle/working/plots")
#   compare(["/kaggle/working/metrics.csv",
#            "/kaggle/working/metrics_zero1.csv",
#            "/kaggle/working/metrics_zero3.csv"], ["DDP", "ZeRO-1", "ZeRO-3"])


Writing zero3_launch.py


In [5]:
!torchrun --nproc_per_node=2 zero3_launch.py

W1002 15:24:48.638000 52 torch/distributed/run.py:852] 
W1002 15:24:48.638000 52 torch/distributed/run.py:852] *****************************************
W1002 15:24:48.638000 52 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 15:24:48.638000 52 torch/distributed/run.py:852] *****************************************
rank 1/2 on cuda:1  
rank 0/2 on cuda:0  


no .txt in data/raw/ to train the BPE onno .txt in data/raw/ to train the BPE on

[rank0]:[W1002 15:24:55.843970376 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
W1002 15:24:58.474000 52 torch/distributed/elastic/multiprocessing/api.py:1010] Sending 

In [6]:
import os
BPE_SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m/bpe"   # folder containing merges.txt from step 1
os.makedirs("/kaggle/working/data", exist_ok=True)
if not os.path.exists("/kaggle/working/data/bpe"):
    os.symlink(BPE_SRC, "/kaggle/working/data/bpe")
print(os.listdir("/kaggle/working/data/bpe"))   # should show vocab.json, merges.txt

['merges.txt', 'vocab.json']


In [7]:
%%writefile zero3_launch.py

# read env vars torchrun injects
import os
# csv metrics logging + step timing
import csv
import time
import math
# add the Kaggle dataset folder to the import path
import sys
# for torch.cuda.set_device
import torch
# collectives: init/all_reduce/broadcast
import torch.distributed as dist
# ZeRO-3 engine: shards params + grads + optimizer, gathers layer-by-layer
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
# gathering the sharded model back into one full state dict for the checkpoint
from torch.distributed.fsdp import StateDictType, FullStateDictConfig
from torch.distributed.fsdp.wrap import ModuleWrapPolicy

# dataset root holds the loose .py files; add it to the import path
SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m"
# shards + bpe live under this subfolder
DATA = f"{SRC}/"
sys.path.insert(0, SRC)
# model/vocab sizes + the data stream factory
from data_pipeline import Config, get_streams
# model + a Block (the unit we shard) + RMSNorm (needs custom init) + helpers
from learn_pipeline import GPT, Block, RMSNorm, make_masks, loss_fn, lr_at


def setup():
    # CPU: read this process's GPU index that torchrun injected (str -> int)
    local_rank = int(os.environ["LOCAL_RANK"])
    # GPU: pin this process to that one GPU (must happen before init)
    torch.cuda.set_device(local_rank)
    # GPU<->GPU: join the process group; device_id tells NCCL our GPU
    dist.init_process_group(
        backend="nccl",
        device_id=torch.device(f"cuda:{local_rank}"),
    )
    # CPU: return identity read back from the group
    return dist.get_rank(), dist.get_world_size(), local_rank


def print0(rank, *args):
    if rank == 0:
        print(*args, flush=True)


def model_info(model, cfg):
    P = sum(p.numel() for p in model.parameters())    # numel works on meta tensors too
    emb = 2 * cfg.V * cfg.D
    return (f"model: {P/1e6:.2f}M params | L={cfg.L} D={cfg.D} H={cfg.H} "
            f"T={cfg.T} V={cfg.V} | embed+head ~{emb/1e6:.1f}M ({100*emb/P:.0f}%)")


def measure_holdings(model, opt):
    # actual bytes from live tensors; under FSDP these are the LOCAL shards (1/N)
    mb = lambda ts: sum(t.numel() * t.element_size() for t in ts) / 1e6
    params_mb = mb(list(model.parameters()))
    grads_mb = mb([p.grad for p in model.parameters() if p.grad is not None])
    optim_mb = mb([t for st in opt.state.values() for t in st.values() if torch.is_tensor(t)])
    return params_mb, grads_mb, optim_mb


def param_init_fn(module):
    # FSDP calls this per module while sharding, to bring META tensors to life:
    # allocate real (uninitialized) tensors on THIS process's GPU...
    module.to_empty(device=torch.cuda.current_device(), recurse=False)
    # ...then set proper initial values (meta tensors carried none)
    with torch.no_grad():
        if isinstance(module, RMSNorm):
            module.g.fill_(1.0)                 # RMSNorm gain starts at 1
        elif hasattr(module, "reset_parameters"):
            module.reset_parameters()           # Linear/Embedding default init


@torch.no_grad()
def evaluate(model, val_stream, device, max_batches):
    """Token-weighted mean loss over up to max_batches of the held-out val set,
    summed across ranks -> a true GLOBAL validation loss (perplexity = e^loss)."""
    model.eval()
    tot = torch.zeros(2, device=device)              # [sum loss*tok, sum tok]
    for i, (x, d) in enumerate(val_stream()):
        if i >= max_batches:
            break
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        attn, loss_mask = make_masks(tokens, doc_id)
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        tot[0] += loss.item() * n
        tot[1] += n
    dist.all_reduce(tot)                             # GPU<->GPU: global sums
    model.train()
    return (tot[0] / tot[1].clamp(min=1)).item()


def main():
    rank, world_size, local_rank = setup()
    device = torch.device(f"cuda:{local_rank}")
    print(f"rank {rank}/{world_size} on cuda:{local_rank}  \n", flush=True)

    # ---- data: disjoint blocks per rank (shared seed) ----
    # CPU: Config builds tokenizer; the stream reads shards into numpy (numpy = CPU RAM)
    cfg = Config()
    # guarantee the ~19.6M ("20M") config no matter which data_pipeline is uploaded
    cfg.D, cfg.H, cfg.L = 384, 8, 7
    cfg.T = 256                                    # longer context (fits a whole story)
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    micro_B = 64                                   # halved for T=256 (attention is O(T^2))
    # ---- token budget: the ONE knob; everything below derives from it ----
    TRAIN_TOKENS = 55_000_000                     # unique train tokens loaded into RAM
    # CPU: sets up the numpy batch iterators (no GPU touched here); keep val too
    train_stream, val_stream = get_streams(cfg, shard_dir=f"{DATA}/shards",
                                       micro_B=micro_B, seed=1234,
                                       train_tokens=TRAIN_TOKENS,
                                       val_tokens=None,
                                       rank=rank, world_size=world_size)

    # STEPS = one pass over the budget: TRAIN_TOKENS / (T * micro_B * world_size)
    # (floor -> never spills into a 2nd epoch; adapts to any world_size)
    STEPS = TRAIN_TOKENS // (cfg.T * micro_B * world_size)
    WARMUP, base_lr, LOG_EVERY = 300, 3e-4, 50
    print0(rank, f"budget {TRAIN_TOKENS/1e6:.0f}M tok | {STEPS} steps x "
                 f"{cfg.T}T x {micro_B}B x {world_size} ranks")
    # val ppl on every log line; 20 batches x B x T x ranks (~164k tok on 2 GPUs)
    # keeps eval overhead ~10-15% of training time (50 batches would be ~30%)
    EVAL_EVERY, EVAL_BATCHES = LOG_EVERY, 20
    torch.manual_seed(0)
    # META init: build the model with SHAPES ONLY -> ZERO bytes allocated, so the full
    # model never lands on any single GPU (this is what avoids OOM for huge models).
    with torch.device("meta"):
        model = GPT(cfg)
    print0(rank, model_info(model, cfg))

    # ---- ZeRO-3: shard params across ranks; FSDP materializes only each rank's 1/N ----
    # GPU: FULL_SHARD = ZeRO-3 (params + grads + optimizer sharded). param_init_fn builds
    # + inits each shard on the GPU, so the whole model is never assembled in one place.
    # (SHARD_GRAD_OP would be ZeRO-2; NO_SHARD would be plain DDP.)
    model = FSDP(
        model,
        # each Block is gathered/freed on its own -> peak memory = ~one Block
        auto_wrap_policy=ModuleWrapPolicy({Block}),
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        device_id=local_rank,
        param_init_fn=param_init_fn,
    )
    # CPU setup over GPU-resident sharded params: MUST be built AFTER FSDP so it sees
    # the sharded (flat) params; its m/v state is allocated on GPU at first step, 1/N size
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr)
    print0(rank, model)

    # one timestamp per run -> every run writes NEW files, never overwrites old ones
    # (only rank 0 writes files, so only rank 0's clock matters)
    RUN_ID = time.strftime("%Y%m%d_%H%M%S")
    OUT_DIR = "/kaggle/working"
    metrics_path = f"{OUT_DIR}/metrics_zero3_{RUN_ID}.csv"
    holdings_path = f"{OUT_DIR}/holdings_zero3_{RUN_ID}.csv"

    if rank == 0:
        print(f"run {RUN_ID} -> {metrics_path}", flush=True)
        logf = open(metrics_path, "w", newline="")
        writer = csv.writer(logf)
        writer.writerow(["step", "tokens", "loss_tokens", "loss", "ppl", "val_loss",
                         "val_ppl", "grad_norm", "tok_per_s", "lr",
                         "alloc_mb", "peak_mb", "reserved_mb", "gpu_used_mb", "gpu_total_mb"])

    # token counters, built from the ACTUAL batch shapes (not assumed T*B)
    local_seen = 0          # this rank: positions fed to the model (B*T per batch)
    local_trained = 0.0     # this rank: positions that counted in the loss (n)
    win_tok, win_t0 = 0, time.time()   # window since last log -> steady tok/s
    for s in range(1, STEPS + 1):
        # CPU: schedule math (plain floats), then push the LR onto the optimizer
        cur_lr = base_lr * lr_at(s, WARMUP, STEPS)
        for g in opt.param_groups:
            g['lr'] = cur_lr
        # CPU: pull one numpy micro-batch off the stream
        x, d = next(train_stream)
        # CPU: verify every sequence is exactly T long, then count what we really got
        assert x.shape == (micro_B, cfg.T) and d.shape == x.shape, \
            f"step {s}: bad batch shape tok={x.shape} doc={d.shape}"
        local_seen += x.shape[0] * x.shape[1]      # rows x sequence length
        win_tok += x.shape[0] * x.shape[1]
        # CPU->GPU: numpy -> GPU tensors (from_numpy is CPU, .to(device) copies over)
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        # GPU: build the attention/loss masks on the data's device
        attn, loss_mask = make_masks(tokens, doc_id)
        # GPU (+ GPU<->GPU comms): FSDP forward all_gathers each Block's full params
        # just-in-time, runs it, then FREES them -> peak memory ~ one Block
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        # GPU: running count of positions that actually got a loss (stays on GPU,
        # no .item() here -> no per-step CPU/GPU sync)
        local_trained += n
        # GPU: drop stale grads
        opt.zero_grad(set_to_none=True)
        # GPU (+ comms): backward gathers params again, computes grads, then
        # reduce_scatters them -> each rank KEEPS ONLY ITS SHARD of the gradients
        loss.backward()
        # GPU (+ comms): FSDP-aware global grad-norm clip; returns the total norm
        gnorm = model.clip_grad_norm_(1.0)
        # GPU: each rank updates ONLY its param shard (no manual all_reduce/broadcast)
        opt.step()

        if s % LOG_EVERY == 0 or s == STEPS:          # always log the final step too
            # GPU<->GPU: one all_reduce for everything we sum across ranks:
            # [loss*n this step, n this step, total seen, total trained, window tokens]
            stats = torch.tensor([loss.item() * float(n), float(n), float(local_seen),
                                  float(local_trained), float(win_tok)],
                                 device=device, dtype=torch.float64)
            dist.all_reduce(stats)
            # GPU->CPU: .item() pulls the scalars back to host for printing/logging
            avg_loss = (stats[0] / stats[1]).item()
            ppl = math.exp(min(avg_loss, 20))            # train perplexity = e^loss
            seen = int(stats[2].item())                  # global positions fed (all ranks)
            trained = int(stats[3].item())               # global positions scored by loss
            # CPU: steady throughput over the whole window, not one noisy step
            win_dt = time.time() - win_t0
            tps = stats[4].item() / max(win_dt, 1e-9)
            eta_min = (STEPS - s) * (win_dt / LOG_EVERY) / 60

            # GPU memory, 4 levels (each contains the one before), read BEFORE eval so
            # the numbers describe training only:
            #   alloc  = tensors alive right now
            #   peak   = highest tensor usage in this window (reset after each log)
            #   resv   = what PyTorch has claimed from the GPU (incl. its free cache)
            #   used   = GPU's own view (like nvidia-smi): + CUDA context + NCCL buffers
            free_b, total_b = torch.cuda.mem_get_info()
            mem = torch.tensor([torch.cuda.memory_allocated(),
                                torch.cuda.max_memory_allocated(),
                                torch.cuda.memory_reserved(),
                                total_b - free_b, total_b],
                               device=device, dtype=torch.float64) / 1e6
            # GPU<->GPU: MAX across ranks -> report the fullest GPU, not just rank 0
            dist.all_reduce(mem, op=dist.ReduceOp.MAX)
            alloc_mb, peak_mb, resv_mb, used_mb, total_mb = mem.tolist()

            # held-out validation perplexity (the real-world quality signal)
            val_loss = val_ppl = ""
            if s % EVAL_EVERY == 0 or s == STEPS:
                vl = evaluate(model, val_stream, device, EVAL_BATCHES)
                val_loss, val_ppl = f"{vl:.4f}", f"{math.exp(min(vl, 20)):.2f}"

            if rank == 0:
                # CPU: write the row to disk
                writer.writerow([s, seen, trained, f"{avg_loss:.4f}", f"{ppl:.2f}",
                                 val_loss, val_ppl, f"{float(gnorm):.2f}",
                                 f"{tps:.0f}", f"{cur_lr:.2e}",
                                 f"{alloc_mb:.0f}", f"{peak_mb:.0f}", f"{resv_mb:.0f}",
                                 f"{used_mb:.0f}", f"{total_mb:.0f}"])
                logf.flush()
                msg = (f"step {s:5d}/{STEPS} | {seen/1e6:6.1f}M tok "
                       f"({trained/1e6:.1f}M in loss) | "
                       f"loss {avg_loss:.4f} ppl {ppl:.1f}")
                if val_ppl:
                    msg += f" | val_ppl {val_ppl}"
                msg += (f" | gnorm {float(gnorm):.2f} | {tps:.0f} tok/s | "
                        f"mem peak {peak_mb:.0f} / resv {resv_mb:.0f} / "
                        f"gpu {used_mb:.0f} of {total_mb:.0f}MB | eta {eta_min:.0f}m")
                print(msg, flush=True)

            # reset the throughput window AND the peak counter AFTER eval/print, so the
            # next window's tok/s and peak memory describe training steps only
            win_tok, win_t0 = 0, time.time()
            torch.cuda.reset_peak_memory_stats()

    # final token audit: counted tokens must equal the budget math
    total = torch.tensor([float(local_seen)], device=device, dtype=torch.float64)
    dist.all_reduce(total)
    expected = STEPS * cfg.T * micro_B * world_size
    print0(rank, f"token audit: counted {int(total.item()):,} | expected {expected:,} | "
                 f"budget {TRAIN_TOKENS:,}")

    if rank == 0:
        logf.close()

    # ---- save a checkpoint for inference ----
    # Under ZeRO-3 no rank holds the whole model, so FSDP must GATHER it first.
    # FULL_STATE_DICT = reassemble every shard into normal (un-sharded) weights with
    # the original parameter names; rank0_only + offload_to_cpu = only rank 0 keeps
    # the result, in CPU RAM (so no GPU ever holds a second full copy).
    # EVERY rank must call state_dict() -- it's a collective (all ranks send shards).
    save_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_cfg):
        full_state = model.state_dict()
    if rank == 0:
        ckpt_path = f"{OUT_DIR}/ckpt_zero3_{RUN_ID}.pt"
        torch.save({
            "model": full_state,
            # the exact shape knobs used here (they were overridden above), so
            # inference rebuilds the SAME architecture without guessing
            "cfg": {"V": cfg.V, "T": cfg.T, "D": cfg.D, "H": cfg.H,
                    "HD": cfg.HD, "DFF": cfg.DFF, "L": cfg.L},
            "run_id": RUN_ID, "steps": STEPS, "train_tokens": TRAIN_TOKENS,
        }, ckpt_path)
        print(f"saved checkpoint -> {ckpt_path}", flush=True)

    # per-rank holdings: params + grads + optim should ALL be ~1/world_size now
    params_mb, grads_mb, optim_mb = measure_holdings(model, opt)
    print(f"rank {rank} holdings: params {params_mb:.0f}MB grads {grads_mb:.0f}MB "
          f"optim {optim_mb:.0f}MB total {params_mb+grads_mb+optim_mb:.0f}MB", flush=True)
    if rank == 0:
        with open(holdings_path, "w", newline="") as hf:
            w = csv.writer(hf)
            w.writerow(["component", "mb"])
            w.writerow(["params", f"{params_mb:.2f}"])
            w.writerow(["grads", f"{grads_mb:.2f}"])
            w.writerow(["optimizer", f"{optim_mb:.2f}"])

    dist.barrier()
    print0(rank, "done")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

# ---- after training, plot / compare (normal notebook cell) ----
#   from plot import plot_holdings, compare
#   (files are timestamped: metrics_zero3_<RUN_ID>.csv -- use the RUN_ID the run printed)
#   plot_holdings("/kaggle/working/holdings_zero3_<RUN_ID>.csv", "/kaggle/working/plots")
#   compare(["/kaggle/working/metrics.csv",
#            "/kaggle/working/metrics_zero1.csv",
#            "/kaggle/working/metrics_zero3_<RUN_ID>.csv"], ["DDP", "ZeRO-1", "ZeRO-3"])

Overwriting zero3_launch.py


In [8]:
!torchrun --nproc_per_node=2 zero3_launch.py

W1002 15:25:00.837000 81 torch/distributed/run.py:852] 
W1002 15:25:00.837000 81 torch/distributed/run.py:852] *****************************************
W1002 15:25:00.837000 81 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 15:25:00.837000 81 torch/distributed/run.py:852] *****************************************
rank 0/2 on cuda:0  
rank 1/2 on cuda:1  


/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
bud

<br>

## Batch Size: 128

In [9]:
%%writefile zero3_launch.py

# read env vars torchrun injects
import os
# csv metrics logging + step timing
import csv
import time
import math
# add the Kaggle dataset folder to the import path
import sys
# for torch.cuda.set_device
import torch
# collectives: init/all_reduce/broadcast
import torch.distributed as dist
# ZeRO-3 engine: shards params + grads + optimizer, gathers layer-by-layer
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
# gathering the sharded model back into one full state dict for the checkpoint
from torch.distributed.fsdp import StateDictType, FullStateDictConfig
from torch.distributed.fsdp.wrap import ModuleWrapPolicy

# dataset root holds the loose .py files; add it to the import path
SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m"
# shards + bpe live under this subfolder
DATA = f"{SRC}/"
sys.path.insert(0, SRC)
# model/vocab sizes + the data stream factory
from data_pipeline import Config, get_streams
# model + a Block (the unit we shard) + RMSNorm (needs custom init) + helpers
from learn_pipeline import GPT, Block, RMSNorm, make_masks, loss_fn, lr_at


def setup():
    # CPU: read this process's GPU index that torchrun injected (str -> int)
    local_rank = int(os.environ["LOCAL_RANK"])
    # GPU: pin this process to that one GPU (must happen before init)
    torch.cuda.set_device(local_rank)
    # GPU<->GPU: join the process group; device_id tells NCCL our GPU
    dist.init_process_group(
        backend="nccl",
        device_id=torch.device(f"cuda:{local_rank}"),
    )
    # CPU: return identity read back from the group
    return dist.get_rank(), dist.get_world_size(), local_rank


def print0(rank, *args):
    if rank == 0:
        print(*args, flush=True)


def model_info(model, cfg):
    P = sum(p.numel() for p in model.parameters())    # numel works on meta tensors too
    emb = 2 * cfg.V * cfg.D
    return (f"model: {P/1e6:.2f}M params | L={cfg.L} D={cfg.D} H={cfg.H} "
            f"T={cfg.T} V={cfg.V} | embed+head ~{emb/1e6:.1f}M ({100*emb/P:.0f}%)")


def measure_holdings(model, opt):
    # actual bytes from live tensors; under FSDP these are the LOCAL shards (1/N)
    mb = lambda ts: sum(t.numel() * t.element_size() for t in ts) / 1e6
    params_mb = mb(list(model.parameters()))
    grads_mb = mb([p.grad for p in model.parameters() if p.grad is not None])
    optim_mb = mb([t for st in opt.state.values() for t in st.values() if torch.is_tensor(t)])
    return params_mb, grads_mb, optim_mb


def param_init_fn(module):
    # FSDP calls this per module while sharding, to bring META tensors to life:
    # allocate real (uninitialized) tensors on THIS process's GPU...
    module.to_empty(device=torch.cuda.current_device(), recurse=False)
    # ...then set proper initial values (meta tensors carried none)
    with torch.no_grad():
        if isinstance(module, RMSNorm):
            module.g.fill_(1.0)                 # RMSNorm gain starts at 1
        elif hasattr(module, "reset_parameters"):
            module.reset_parameters()           # Linear/Embedding default init


@torch.no_grad()
def evaluate(model, val_stream, device, max_batches):
    """Token-weighted mean loss over up to max_batches of the held-out val set,
    summed across ranks -> a true GLOBAL validation loss (perplexity = e^loss)."""
    model.eval()
    tot = torch.zeros(2, device=device)              # [sum loss*tok, sum tok]
    for i, (x, d) in enumerate(val_stream()):
        if i >= max_batches:
            break
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        attn, loss_mask = make_masks(tokens, doc_id)
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        tot[0] += loss.item() * n
        tot[1] += n
    dist.all_reduce(tot)                             # GPU<->GPU: global sums
    model.train()
    return (tot[0] / tot[1].clamp(min=1)).item()


def main():
    rank, world_size, local_rank = setup()
    device = torch.device(f"cuda:{local_rank}")
    print(f"rank {rank}/{world_size} on cuda:{local_rank}  \n", flush=True)

    # ---- data: disjoint blocks per rank (shared seed) ----
    # CPU: Config builds tokenizer; the stream reads shards into numpy (numpy = CPU RAM)
    cfg = Config()
    # guarantee the ~19.6M ("20M") config no matter which data_pipeline is uploaded
    cfg.D, cfg.H, cfg.L = 384, 8, 7
    cfg.T = 256                                    # longer context (fits a whole story)
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    micro_B = 128                                   # halved for T=256 (attention is O(T^2))
    # ---- token budget: the ONE knob; everything below derives from it ----
    TRAIN_TOKENS = 55_000_000                     # unique train tokens loaded into RAM
    # CPU: sets up the numpy batch iterators (no GPU touched here); keep val too
    train_stream, val_stream = get_streams(cfg, shard_dir=f"{DATA}/shards",
                                       micro_B=micro_B, seed=1234,
                                       train_tokens=TRAIN_TOKENS,
                                       val_tokens=None,
                                       rank=rank, world_size=world_size)

    # STEPS = one pass over the budget: TRAIN_TOKENS / (T * micro_B * world_size)
    # (floor -> never spills into a 2nd epoch; adapts to any world_size)
    STEPS = TRAIN_TOKENS // (cfg.T * micro_B * world_size)
    WARMUP, base_lr, LOG_EVERY = 300, 3e-4, 50
    print0(rank, f"budget {TRAIN_TOKENS/1e6:.0f}M tok | {STEPS} steps x "
                 f"{cfg.T}T x {micro_B}B x {world_size} ranks")
    # val ppl on every log line; 20 batches x B x T x ranks (~164k tok on 2 GPUs)
    # keeps eval overhead ~10-15% of training time (50 batches would be ~30%)
    EVAL_EVERY, EVAL_BATCHES = LOG_EVERY, 20
    torch.manual_seed(0)
    # META init: build the model with SHAPES ONLY -> ZERO bytes allocated, so the full
    # model never lands on any single GPU (this is what avoids OOM for huge models).
    with torch.device("meta"):
        model = GPT(cfg)
    print0(rank, model_info(model, cfg))

    # ---- ZeRO-3: shard params across ranks; FSDP materializes only each rank's 1/N ----
    # GPU: FULL_SHARD = ZeRO-3 (params + grads + optimizer sharded). param_init_fn builds
    # + inits each shard on the GPU, so the whole model is never assembled in one place.
    # (SHARD_GRAD_OP would be ZeRO-2; NO_SHARD would be plain DDP.)
    model = FSDP(
        model,
        # each Block is gathered/freed on its own -> peak memory = ~one Block
        auto_wrap_policy=ModuleWrapPolicy({Block}),
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        device_id=local_rank,
        param_init_fn=param_init_fn,
    )
    # CPU setup over GPU-resident sharded params: MUST be built AFTER FSDP so it sees
    # the sharded (flat) params; its m/v state is allocated on GPU at first step, 1/N size
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr)
    print0(rank, model)

    # one timestamp per run -> every run writes NEW files, never overwrites old ones
    # (only rank 0 writes files, so only rank 0's clock matters)
    RUN_ID = time.strftime("%Y%m%d_%H%M%S")
    OUT_DIR = "/kaggle/working"
    metrics_path = f"{OUT_DIR}/metrics_zero3_{RUN_ID}.csv"
    holdings_path = f"{OUT_DIR}/holdings_zero3_{RUN_ID}.csv"

    if rank == 0:
        print(f"run {RUN_ID} -> {metrics_path}", flush=True)
        logf = open(metrics_path, "w", newline="")
        writer = csv.writer(logf)
        writer.writerow(["step", "tokens", "loss_tokens", "loss", "ppl", "val_loss",
                         "val_ppl", "grad_norm", "tok_per_s", "lr",
                         "alloc_mb", "peak_mb", "reserved_mb", "gpu_used_mb", "gpu_total_mb"])

    # token counters, built from the ACTUAL batch shapes (not assumed T*B)
    local_seen = 0          # this rank: positions fed to the model (B*T per batch)
    local_trained = 0.0     # this rank: positions that counted in the loss (n)
    win_tok, win_t0 = 0, time.time()   # window since last log -> steady tok/s
    for s in range(1, STEPS + 1):
        # CPU: schedule math (plain floats), then push the LR onto the optimizer
        cur_lr = base_lr * lr_at(s, WARMUP, STEPS)
        for g in opt.param_groups:
            g['lr'] = cur_lr
        # CPU: pull one numpy micro-batch off the stream
        x, d = next(train_stream)
        # CPU: verify every sequence is exactly T long, then count what we really got
        assert x.shape == (micro_B, cfg.T) and d.shape == x.shape, \
            f"step {s}: bad batch shape tok={x.shape} doc={d.shape}"
        local_seen += x.shape[0] * x.shape[1]      # rows x sequence length
        win_tok += x.shape[0] * x.shape[1]
        # CPU->GPU: numpy -> GPU tensors (from_numpy is CPU, .to(device) copies over)
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        # GPU: build the attention/loss masks on the data's device
        attn, loss_mask = make_masks(tokens, doc_id)
        # GPU (+ GPU<->GPU comms): FSDP forward all_gathers each Block's full params
        # just-in-time, runs it, then FREES them -> peak memory ~ one Block
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        # GPU: running count of positions that actually got a loss (stays on GPU,
        # no .item() here -> no per-step CPU/GPU sync)
        local_trained += n
        # GPU: drop stale grads
        opt.zero_grad(set_to_none=True)
        # GPU (+ comms): backward gathers params again, computes grads, then
        # reduce_scatters them -> each rank KEEPS ONLY ITS SHARD of the gradients
        loss.backward()
        # GPU (+ comms): FSDP-aware global grad-norm clip; returns the total norm
        gnorm = model.clip_grad_norm_(1.0)
        # GPU: each rank updates ONLY its param shard (no manual all_reduce/broadcast)
        opt.step()

        if s % LOG_EVERY == 0 or s == STEPS:          # always log the final step too
            # GPU<->GPU: one all_reduce for everything we sum across ranks:
            # [loss*n this step, n this step, total seen, total trained, window tokens]
            stats = torch.tensor([loss.item() * float(n), float(n), float(local_seen),
                                  float(local_trained), float(win_tok)],
                                 device=device, dtype=torch.float64)
            dist.all_reduce(stats)
            # GPU->CPU: .item() pulls the scalars back to host for printing/logging
            avg_loss = (stats[0] / stats[1]).item()
            ppl = math.exp(min(avg_loss, 20))            # train perplexity = e^loss
            seen = int(stats[2].item())                  # global positions fed (all ranks)
            trained = int(stats[3].item())               # global positions scored by loss
            # CPU: steady throughput over the whole window, not one noisy step
            win_dt = time.time() - win_t0
            tps = stats[4].item() / max(win_dt, 1e-9)
            eta_min = (STEPS - s) * (win_dt / LOG_EVERY) / 60

            # GPU memory, 4 levels (each contains the one before), read BEFORE eval so
            # the numbers describe training only:
            #   alloc  = tensors alive right now
            #   peak   = highest tensor usage in this window (reset after each log)
            #   resv   = what PyTorch has claimed from the GPU (incl. its free cache)
            #   used   = GPU's own view (like nvidia-smi): + CUDA context + NCCL buffers
            free_b, total_b = torch.cuda.mem_get_info()
            mem = torch.tensor([torch.cuda.memory_allocated(),
                                torch.cuda.max_memory_allocated(),
                                torch.cuda.memory_reserved(),
                                total_b - free_b, total_b],
                               device=device, dtype=torch.float64) / 1e6
            # GPU<->GPU: MAX across ranks -> report the fullest GPU, not just rank 0
            dist.all_reduce(mem, op=dist.ReduceOp.MAX)
            alloc_mb, peak_mb, resv_mb, used_mb, total_mb = mem.tolist()

            # held-out validation perplexity (the real-world quality signal)
            val_loss = val_ppl = ""
            if s % EVAL_EVERY == 0 or s == STEPS:
                vl = evaluate(model, val_stream, device, EVAL_BATCHES)
                val_loss, val_ppl = f"{vl:.4f}", f"{math.exp(min(vl, 20)):.2f}"

            if rank == 0:
                # CPU: write the row to disk
                writer.writerow([s, seen, trained, f"{avg_loss:.4f}", f"{ppl:.2f}",
                                 val_loss, val_ppl, f"{float(gnorm):.2f}",
                                 f"{tps:.0f}", f"{cur_lr:.2e}",
                                 f"{alloc_mb:.0f}", f"{peak_mb:.0f}", f"{resv_mb:.0f}",
                                 f"{used_mb:.0f}", f"{total_mb:.0f}"])
                logf.flush()
                msg = (f"step {s:5d}/{STEPS} | {seen/1e6:6.1f}M tok "
                       f"({trained/1e6:.1f}M in loss) | "
                       f"loss {avg_loss:.4f} ppl {ppl:.1f}")
                if val_ppl:
                    msg += f" | val_ppl {val_ppl}"
                msg += (f" | gnorm {float(gnorm):.2f} | {tps:.0f} tok/s | "
                        f"mem peak {peak_mb:.0f} / resv {resv_mb:.0f} / "
                        f"gpu {used_mb:.0f} of {total_mb:.0f}MB | eta {eta_min:.0f}m")
                print(msg, flush=True)

            # reset the throughput window AND the peak counter AFTER eval/print, so the
            # next window's tok/s and peak memory describe training steps only
            win_tok, win_t0 = 0, time.time()
            torch.cuda.reset_peak_memory_stats()

    # final token audit: counted tokens must equal the budget math
    total = torch.tensor([float(local_seen)], device=device, dtype=torch.float64)
    dist.all_reduce(total)
    expected = STEPS * cfg.T * micro_B * world_size
    print0(rank, f"token audit: counted {int(total.item()):,} | expected {expected:,} | "
                 f"budget {TRAIN_TOKENS:,}")

    if rank == 0:
        logf.close()

    # ---- save a checkpoint for inference ----
    # Under ZeRO-3 no rank holds the whole model, so FSDP must GATHER it first.
    # FULL_STATE_DICT = reassemble every shard into normal (un-sharded) weights with
    # the original parameter names; rank0_only + offload_to_cpu = only rank 0 keeps
    # the result, in CPU RAM (so no GPU ever holds a second full copy).
    # EVERY rank must call state_dict() -- it's a collective (all ranks send shards).
    save_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_cfg):
        full_state = model.state_dict()
    if rank == 0:
        ckpt_path = f"{OUT_DIR}/ckpt_zero3_{RUN_ID}.pt"
        torch.save({
            "model": full_state,
            # the exact shape knobs used here (they were overridden above), so
            # inference rebuilds the SAME architecture without guessing
            "cfg": {"V": cfg.V, "T": cfg.T, "D": cfg.D, "H": cfg.H,
                    "HD": cfg.HD, "DFF": cfg.DFF, "L": cfg.L},
            "run_id": RUN_ID, "steps": STEPS, "train_tokens": TRAIN_TOKENS,
        }, ckpt_path)
        print(f"saved checkpoint -> {ckpt_path}", flush=True)

    # per-rank holdings: params + grads + optim should ALL be ~1/world_size now
    params_mb, grads_mb, optim_mb = measure_holdings(model, opt)
    print(f"rank {rank} holdings: params {params_mb:.0f}MB grads {grads_mb:.0f}MB "
          f"optim {optim_mb:.0f}MB total {params_mb+grads_mb+optim_mb:.0f}MB", flush=True)
    if rank == 0:
        with open(holdings_path, "w", newline="") as hf:
            w = csv.writer(hf)
            w.writerow(["component", "mb"])
            w.writerow(["params", f"{params_mb:.2f}"])
            w.writerow(["grads", f"{grads_mb:.2f}"])
            w.writerow(["optimizer", f"{optim_mb:.2f}"])

    dist.barrier()
    print0(rank, "done")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

# ---- after training, plot / compare (normal notebook cell) ----
#   from plot import plot_holdings, compare
#   (files are timestamped: metrics_zero3_<RUN_ID>.csv -- use the RUN_ID the run printed)
#   plot_holdings("/kaggle/working/holdings_zero3_<RUN_ID>.csv", "/kaggle/working/plots")
#   compare(["/kaggle/working/metrics.csv",
#            "/kaggle/working/metrics_zero1.csv",
#            "/kaggle/working/metrics_zero3_<RUN_ID>.csv"], ["DDP", "ZeRO-1", "ZeRO-3"])

Overwriting zero3_launch.py


In [10]:
!torchrun --nproc_per_node=2 zero3_launch.py

W1002 15:53:22.551000 114 torch/distributed/run.py:852] 
W1002 15:53:22.551000 114 torch/distributed/run.py:852] *****************************************
W1002 15:53:22.551000 114 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 15:53:22.551000 114 torch/distributed/run.py:852] *****************************************
rank 1/2 on cuda:1  

rank 0/2 on cuda:0  

/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks


## Batch Size : 120

In [11]:
%%writefile zero3_launch.py

# read env vars torchrun injects
import os
# csv metrics logging + step timing
import csv
import time
import math
# add the Kaggle dataset folder to the import path
import sys
# for torch.cuda.set_device
import torch
# collectives: init/all_reduce/broadcast
import torch.distributed as dist
# ZeRO-3 engine: shards params + grads + optimizer, gathers layer-by-layer
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy
# gathering the sharded model back into one full state dict for the checkpoint
from torch.distributed.fsdp import StateDictType, FullStateDictConfig
from torch.distributed.fsdp.wrap import ModuleWrapPolicy

# dataset root holds the loose .py files; add it to the import path
SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m"
# shards + bpe live under this subfolder
DATA = f"{SRC}/"
sys.path.insert(0, SRC)
# model/vocab sizes + the data stream factory
from data_pipeline import Config, get_streams
# model + a Block (the unit we shard) + RMSNorm (needs custom init) + helpers
from learn_pipeline import GPT, Block, RMSNorm, make_masks, loss_fn, lr_at


def setup():
    # CPU: read this process's GPU index that torchrun injected (str -> int)
    local_rank = int(os.environ["LOCAL_RANK"])
    # GPU: pin this process to that one GPU (must happen before init)
    torch.cuda.set_device(local_rank)
    # GPU<->GPU: join the process group; device_id tells NCCL our GPU
    dist.init_process_group(
        backend="nccl",
        device_id=torch.device(f"cuda:{local_rank}"),
    )
    # CPU: return identity read back from the group
    return dist.get_rank(), dist.get_world_size(), local_rank


def print0(rank, *args):
    if rank == 0:
        print(*args, flush=True)


def model_info(model, cfg):
    P = sum(p.numel() for p in model.parameters())    # numel works on meta tensors too
    emb = 2 * cfg.V * cfg.D
    return (f"model: {P/1e6:.2f}M params | L={cfg.L} D={cfg.D} H={cfg.H} "
            f"T={cfg.T} V={cfg.V} | embed+head ~{emb/1e6:.1f}M ({100*emb/P:.0f}%)")


def measure_holdings(model, opt):
    # actual bytes from live tensors; under FSDP these are the LOCAL shards (1/N)
    mb = lambda ts: sum(t.numel() * t.element_size() for t in ts) / 1e6
    params_mb = mb(list(model.parameters()))
    grads_mb = mb([p.grad for p in model.parameters() if p.grad is not None])
    optim_mb = mb([t for st in opt.state.values() for t in st.values() if torch.is_tensor(t)])
    return params_mb, grads_mb, optim_mb


def param_init_fn(module):
    # FSDP calls this per module while sharding, to bring META tensors to life:
    # allocate real (uninitialized) tensors on THIS process's GPU...
    module.to_empty(device=torch.cuda.current_device(), recurse=False)
    # ...then set proper initial values (meta tensors carried none)
    with torch.no_grad():
        if isinstance(module, RMSNorm):
            module.g.fill_(1.0)                 # RMSNorm gain starts at 1
        elif hasattr(module, "reset_parameters"):
            module.reset_parameters()           # Linear/Embedding default init


@torch.no_grad()
def evaluate(model, val_stream, device, max_batches):
    """Token-weighted mean loss over up to max_batches of the held-out val set,
    summed across ranks -> a true GLOBAL validation loss (perplexity = e^loss)."""
    model.eval()
    tot = torch.zeros(2, device=device)              # [sum loss*tok, sum tok]
    for i, (x, d) in enumerate(val_stream()):
        if i >= max_batches:
            break
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        attn, loss_mask = make_masks(tokens, doc_id)
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        tot[0] += loss.item() * n
        tot[1] += n
    dist.all_reduce(tot)                             # GPU<->GPU: global sums
    model.train()
    return (tot[0] / tot[1].clamp(min=1)).item()


def main():
    rank, world_size, local_rank = setup()
    device = torch.device(f"cuda:{local_rank}")
    print(f"rank {rank}/{world_size} on cuda:{local_rank}  \n", flush=True)

    # ---- data: disjoint blocks per rank (shared seed) ----
    # CPU: Config builds tokenizer; the stream reads shards into numpy (numpy = CPU RAM)
    cfg = Config()
    # guarantee the ~19.6M ("20M") config no matter which data_pipeline is uploaded
    cfg.D, cfg.H, cfg.L = 384, 8, 7
    cfg.T = 256                                    # longer context (fits a whole story)
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    micro_B = 120                                   # halved for T=256 (attention is O(T^2))
    # ---- token budget: the ONE knob; everything below derives from it ----
    TRAIN_TOKENS = 55_000_000                     # unique train tokens loaded into RAM
    # CPU: sets up the numpy batch iterators (no GPU touched here); keep val too
    train_stream, val_stream = get_streams(cfg, shard_dir=f"{DATA}/shards",
                                       micro_B=micro_B, seed=1234,
                                       train_tokens=TRAIN_TOKENS,
                                       val_tokens=None,
                                       rank=rank, world_size=world_size)

    # STEPS = one pass over the budget: TRAIN_TOKENS / (T * micro_B * world_size)
    # (floor -> never spills into a 2nd epoch; adapts to any world_size)
    STEPS = TRAIN_TOKENS // (cfg.T * micro_B * world_size)
    WARMUP, base_lr, LOG_EVERY = 300, 3e-4, 50
    print0(rank, f"budget {TRAIN_TOKENS/1e6:.0f}M tok | {STEPS} steps x "
                 f"{cfg.T}T x {micro_B}B x {world_size} ranks")
    # val ppl on every log line; 20 batches x B x T x ranks (~164k tok on 2 GPUs)
    # keeps eval overhead ~10-15% of training time (50 batches would be ~30%)
    EVAL_EVERY, EVAL_BATCHES = LOG_EVERY, 20
    torch.manual_seed(0)
    # META init: build the model with SHAPES ONLY -> ZERO bytes allocated, so the full
    # model never lands on any single GPU (this is what avoids OOM for huge models).
    with torch.device("meta"):
        model = GPT(cfg)
    print0(rank, model_info(model, cfg))

    # ---- ZeRO-3: shard params across ranks; FSDP materializes only each rank's 1/N ----
    # GPU: FULL_SHARD = ZeRO-3 (params + grads + optimizer sharded). param_init_fn builds
    # + inits each shard on the GPU, so the whole model is never assembled in one place.
    # (SHARD_GRAD_OP would be ZeRO-2; NO_SHARD would be plain DDP.)
    model = FSDP(
        model,
        # each Block is gathered/freed on its own -> peak memory = ~one Block
        auto_wrap_policy=ModuleWrapPolicy({Block}),
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        device_id=local_rank,
        param_init_fn=param_init_fn,
    )
    # CPU setup over GPU-resident sharded params: MUST be built AFTER FSDP so it sees
    # the sharded (flat) params; its m/v state is allocated on GPU at first step, 1/N size
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr)
    print0(rank, model)

    # one timestamp per run -> every run writes NEW files, never overwrites old ones
    # (only rank 0 writes files, so only rank 0's clock matters)
    RUN_ID = time.strftime("%Y%m%d_%H%M%S")
    OUT_DIR = "/kaggle/working"
    metrics_path = f"{OUT_DIR}/metrics_zero3_{RUN_ID}.csv"
    holdings_path = f"{OUT_DIR}/holdings_zero3_{RUN_ID}.csv"

    if rank == 0:
        print(f"run {RUN_ID} -> {metrics_path}", flush=True)
        logf = open(metrics_path, "w", newline="")
        writer = csv.writer(logf)
        writer.writerow(["step", "tokens", "loss_tokens", "loss", "ppl", "val_loss",
                         "val_ppl", "grad_norm", "tok_per_s", "lr",
                         "alloc_mb", "peak_mb", "reserved_mb", "gpu_used_mb", "gpu_total_mb"])

    # token counters, built from the ACTUAL batch shapes (not assumed T*B)
    local_seen = 0          # this rank: positions fed to the model (B*T per batch)
    local_trained = 0.0     # this rank: positions that counted in the loss (n)
    win_tok, win_t0 = 0, time.time()   # window since last log -> steady tok/s
    for s in range(1, STEPS + 1):
        # CPU: schedule math (plain floats), then push the LR onto the optimizer
        cur_lr = base_lr * lr_at(s, WARMUP, STEPS)
        for g in opt.param_groups:
            g['lr'] = cur_lr
        # CPU: pull one numpy micro-batch off the stream
        x, d = next(train_stream)
        # CPU: verify every sequence is exactly T long, then count what we really got
        assert x.shape == (micro_B, cfg.T) and d.shape == x.shape, \
            f"step {s}: bad batch shape tok={x.shape} doc={d.shape}"
        local_seen += x.shape[0] * x.shape[1]      # rows x sequence length
        win_tok += x.shape[0] * x.shape[1]
        # CPU->GPU: numpy -> GPU tensors (from_numpy is CPU, .to(device) copies over)
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        # GPU: build the attention/loss masks on the data's device
        attn, loss_mask = make_masks(tokens, doc_id)
        # GPU (+ GPU<->GPU comms): FSDP forward all_gathers each Block's full params
        # just-in-time, runs it, then FREES them -> peak memory ~ one Block
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        # GPU: running count of positions that actually got a loss (stays on GPU,
        # no .item() here -> no per-step CPU/GPU sync)
        local_trained += n
        # GPU: drop stale grads
        opt.zero_grad(set_to_none=True)
        # GPU (+ comms): backward gathers params again, computes grads, then
        # reduce_scatters them -> each rank KEEPS ONLY ITS SHARD of the gradients
        loss.backward()
        # GPU (+ comms): FSDP-aware global grad-norm clip; returns the total norm
        gnorm = model.clip_grad_norm_(1.0)
        # GPU: each rank updates ONLY its param shard (no manual all_reduce/broadcast)
        opt.step()

        if s % LOG_EVERY == 0 or s == STEPS:          # always log the final step too
            # GPU<->GPU: one all_reduce for everything we sum across ranks:
            # [loss*n this step, n this step, total seen, total trained, window tokens]
            stats = torch.tensor([loss.item() * float(n), float(n), float(local_seen),
                                  float(local_trained), float(win_tok)],
                                 device=device, dtype=torch.float64)
            dist.all_reduce(stats)
            # GPU->CPU: .item() pulls the scalars back to host for printing/logging
            avg_loss = (stats[0] / stats[1]).item()
            ppl = math.exp(min(avg_loss, 20))            # train perplexity = e^loss
            seen = int(stats[2].item())                  # global positions fed (all ranks)
            trained = int(stats[3].item())               # global positions scored by loss
            # CPU: steady throughput over the whole window, not one noisy step
            win_dt = time.time() - win_t0
            tps = stats[4].item() / max(win_dt, 1e-9)
            eta_min = (STEPS - s) * (win_dt / LOG_EVERY) / 60

            # GPU memory, 4 levels (each contains the one before), read BEFORE eval so
            # the numbers describe training only:
            #   alloc  = tensors alive right now
            #   peak   = highest tensor usage in this window (reset after each log)
            #   resv   = what PyTorch has claimed from the GPU (incl. its free cache)
            #   used   = GPU's own view (like nvidia-smi): + CUDA context + NCCL buffers
            free_b, total_b = torch.cuda.mem_get_info()
            mem = torch.tensor([torch.cuda.memory_allocated(),
                                torch.cuda.max_memory_allocated(),
                                torch.cuda.memory_reserved(),
                                total_b - free_b, total_b],
                               device=device, dtype=torch.float64) / 1e6
            # GPU<->GPU: MAX across ranks -> report the fullest GPU, not just rank 0
            dist.all_reduce(mem, op=dist.ReduceOp.MAX)
            alloc_mb, peak_mb, resv_mb, used_mb, total_mb = mem.tolist()

            # held-out validation perplexity (the real-world quality signal)
            val_loss = val_ppl = ""
            if s % EVAL_EVERY == 0 or s == STEPS:
                vl = evaluate(model, val_stream, device, EVAL_BATCHES)
                val_loss, val_ppl = f"{vl:.4f}", f"{math.exp(min(vl, 20)):.2f}"

            if rank == 0:
                # CPU: write the row to disk
                writer.writerow([s, seen, trained, f"{avg_loss:.4f}", f"{ppl:.2f}",
                                 val_loss, val_ppl, f"{float(gnorm):.2f}",
                                 f"{tps:.0f}", f"{cur_lr:.2e}",
                                 f"{alloc_mb:.0f}", f"{peak_mb:.0f}", f"{resv_mb:.0f}",
                                 f"{used_mb:.0f}", f"{total_mb:.0f}"])
                logf.flush()
                msg = (f"step {s:5d}/{STEPS} | {seen/1e6:6.1f}M tok "
                       f"({trained/1e6:.1f}M in loss) | "
                       f"loss {avg_loss:.4f} ppl {ppl:.1f}")
                if val_ppl:
                    msg += f" | val_ppl {val_ppl}"
                msg += (f" | gnorm {float(gnorm):.2f} | {tps:.0f} tok/s | "
                        f"mem peak {peak_mb:.0f} / resv {resv_mb:.0f} / "
                        f"gpu {used_mb:.0f} of {total_mb:.0f}MB | eta {eta_min:.0f}m")
                print(msg, flush=True)

            # reset the throughput window AND the peak counter AFTER eval/print, so the
            # next window's tok/s and peak memory describe training steps only
            win_tok, win_t0 = 0, time.time()
            torch.cuda.reset_peak_memory_stats()

    # final token audit: counted tokens must equal the budget math
    total = torch.tensor([float(local_seen)], device=device, dtype=torch.float64)
    dist.all_reduce(total)
    expected = STEPS * cfg.T * micro_B * world_size
    print0(rank, f"token audit: counted {int(total.item()):,} | expected {expected:,} | "
                 f"budget {TRAIN_TOKENS:,}")

    if rank == 0:
        logf.close()

    # ---- save a checkpoint for inference ----
    # Under ZeRO-3 no rank holds the whole model, so FSDP must GATHER it first.
    # FULL_STATE_DICT = reassemble every shard into normal (un-sharded) weights with
    # the original parameter names; rank0_only + offload_to_cpu = only rank 0 keeps
    # the result, in CPU RAM (so no GPU ever holds a second full copy).
    # EVERY rank must call state_dict() -- it's a collective (all ranks send shards).
    save_cfg = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, save_cfg):
        full_state = model.state_dict()
    if rank == 0:
        ckpt_path = f"{OUT_DIR}/ckpt_zero3_{RUN_ID}.pt"
        torch.save({
            "model": full_state,
            # the exact shape knobs used here (they were overridden above), so
            # inference rebuilds the SAME architecture without guessing
            "cfg": {"V": cfg.V, "T": cfg.T, "D": cfg.D, "H": cfg.H,
                    "HD": cfg.HD, "DFF": cfg.DFF, "L": cfg.L},
            "run_id": RUN_ID, "steps": STEPS, "train_tokens": TRAIN_TOKENS,
        }, ckpt_path)
        print(f"saved checkpoint -> {ckpt_path}", flush=True)

    # per-rank holdings: params + grads + optim should ALL be ~1/world_size now
    params_mb, grads_mb, optim_mb = measure_holdings(model, opt)
    print(f"rank {rank} holdings: params {params_mb:.0f}MB grads {grads_mb:.0f}MB "
          f"optim {optim_mb:.0f}MB total {params_mb+grads_mb+optim_mb:.0f}MB", flush=True)
    if rank == 0:
        with open(holdings_path, "w", newline="") as hf:
            w = csv.writer(hf)
            w.writerow(["component", "mb"])
            w.writerow(["params", f"{params_mb:.2f}"])
            w.writerow(["grads", f"{grads_mb:.2f}"])
            w.writerow(["optimizer", f"{optim_mb:.2f}"])

    dist.barrier()
    print0(rank, "done")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

# ---- after training, plot / compare (normal notebook cell) ----
#   from plot import plot_holdings, compare
#   (files are timestamped: metrics_zero3_<RUN_ID>.csv -- use the RUN_ID the run printed)
#   plot_holdings("/kaggle/working/holdings_zero3_<RUN_ID>.csv", "/kaggle/working/plots")
#   compare(["/kaggle/working/metrics.csv",
#            "/kaggle/working/metrics_zero1.csv",
#            "/kaggle/working/metrics_zero3_<RUN_ID>.csv"], ["DDP", "ZeRO-1", "ZeRO-3"])

Overwriting zero3_launch.py


In [12]:
!torchrun --nproc_per_node=2 zero3_launch.py

W1002 15:53:33.101000 145 torch/distributed/run.py:852] 
W1002 15:53:33.101000 145 torch/distributed/run.py:852] *****************************************
W1002 15:53:33.101000 145 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 15:53:33.101000 145 torch/distributed/run.py:852] *****************************************
rank 1/2 on cuda:1  

rank 0/2 on cuda:0  

/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m//shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks

## With Reversibility

In [13]:
%%writefile data_pipeline.py

"""
data_pipeline.py -- PIPELINE A: raw documents -> token shards (run ONCE).

Responsibilities:
  SETUP    Tokenizer, Config (shared vocab/model sizes)
  STAGE 0  RawCorpus   -- load raw text documents from a folder
  STAGE 1  ShardWriter -- tokenize docs -> token shards saved to disk
  STAGE 2  ShardStream -- load a token budget of shards into RAM -> batches

Run once (offline):
    python data_pipeline.py

Tokens are pre-tokenized and SAVED here, never tokenized on the fly.
learn_pipeline.py starts from the shards this produces.
"""

import os
import glob

import numpy as np
from tokenizers import ByteLevelBPETokenizer


# where the trained BPE lives, what it trains on, and the target vocab size
BPE_DIR = "data/bpe"
RAW_DIR = "data/raw"
VOCAB_SIZE = 4000        # target; actual may be smaller on a tiny corpus


# ---- SETUP -----------------------------------------------------------------
class Tokenizer:
    """Byte-level BPE trained on YOUR corpus (small vocab -> small embeddings).

    First construction trains the BPE from RAW_DIR and saves it to BPE_DIR;
    later constructions just load it. PAD/EOS are reserved ids just past the
    learned vocab (so the model sizes its embedding to V = vocab + 2).
    """

    def __init__(self, bpe_dir=BPE_DIR, raw_dir=RAW_DIR, vocab_size=VOCAB_SIZE):
        vocab_json = os.path.join(bpe_dir, "vocab.json")
        merges_txt = os.path.join(bpe_dir, "merges.txt")
        if not (os.path.exists(vocab_json) and os.path.exists(merges_txt)):
            self._train(bpe_dir, raw_dir, vocab_size)
        self.bpe = ByteLevelBPETokenizer(vocab_json, merges_txt)
        n = self.bpe.get_vocab_size()
        self.PAD = n          # reserved ids just past the learned vocab
        self.EOS = n + 1
        self.V = n + 2        # total vocab the model must size its embedding to

    def _train(self, bpe_dir, raw_dir, vocab_size):
        files = sorted(glob.glob(os.path.join(raw_dir, "*.txt")))
        if not files:
            raise SystemExit(f"no .txt in {raw_dir}/ to train the BPE on")
        total_mb = sum(os.path.getsize(f) for f in files) / 1e6
        print(f"[BPE] training on {len(files)} files ({total_mb:.1f} MB), "
              f"target vocab {vocab_size} ...", flush=True)
        os.makedirs(bpe_dir, exist_ok=True)
        bpe = ByteLevelBPETokenizer()
        # min_frequency=1 lets it keep merging until it HITS vocab_size exactly
        bpe.train(files=files, vocab_size=vocab_size, min_frequency=1)
        bpe.save_model(bpe_dir)
        got = bpe.get_vocab_size()
        print(f"[BPE] done: {got} tokens (target {vocab_size}) -> {bpe_dir}/", flush=True)
        if got < vocab_size:
            print(f"[BPE] WARNING: only {got} < {vocab_size} -- corpus too small", flush=True)

    def encode(self, text):
        return self.bpe.encode(text).ids

    def decode(self, ids):
        ids = [i for i in ids if i not in (self.PAD, self.EOS)]
        return self.bpe.decode(ids)

    def piece(self, i):
        if i == self.PAD: return "<pad>"
        if i == self.EOS: return "<eos>"
        return self.bpe.decode([i])


class Config:
    def __init__(self):
        self.TOK = Tokenizer()
        self.V = self.TOK.V  # vocabulary size (from the BPE tokenizer)
        self.T = 256     # context window (fits a whole TinyStory ~200 tok)
        self.D = 256     # model dimension
        self.H = 8      # number of attention heads
        self.HD = self.D // self.H  # head dimension
        self.DFF = self.D*4  # feed-forward dimension
        self.L = 6      # number of layers


# ---- STAGE 0: load raw documents ------------------------------------------
class RawCorpus:
    """A folder of raw text files; each file is one document.

    Drop your own .txt files into `folder` (one document per file) and the
    rest of the pipeline picks them up. docs() streams files lazily so the
    whole corpus never needs to sit in RAM.
    """

    def __init__(self, folder, pattern="*.txt"):
        self.folder = folder
        self.pattern = pattern

    def paths(self):
        return sorted(glob.glob(os.path.join(self.folder, self.pattern)))

    def docs(self):
        for p in self.paths():
            with open(p, "r", encoding="utf-8") as f:
                yield f.read()

    def __len__(self):
        return len(self.paths())


# ---- STAGE 1: tokenize -> token shards on disk (run once) -----------------
class ShardWriter:
    """Tokenize a corpus -> fixed-size train/val shards.

    Each shard i -> two parallel files under out_dir/<split>/:
        shard_{i}.tok.npy  token ids (uint16)
        shard_{i}.doc.npy  doc ids   (uint32)  -> lets the trainer rebuild the
                                                   cross-document attention mask.
    Real pipelines use ~100M tokens/shard; we keep it tiny to see the mechanics.
    """

    TOKEN_DTYPE = np.uint16
    DOCID_DTYPE = np.uint32

    def __init__(self, tokenizer, out_dir="data/shards", shard_size=1000,
                 val_frac=0.1):
        self.tok = tokenizer
        self.out_dir = out_dir
        self.shard_size = shard_size
        self.val_frac = val_frac

    def _write_split(self, docs, split):
        split_dir = os.path.join(self.out_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        tok_buf, doc_buf, idx = [], [], 0

        def flush(tok_buf, doc_buf, idx):
            base = os.path.join(split_dir, f"shard_{idx:05d}")
            np.save(base + ".tok.npy", np.array(tok_buf, dtype=self.TOKEN_DTYPE))
            np.save(base + ".doc.npy", np.array(doc_buf, dtype=self.DOCID_DTYPE))
            print(f"  {split}/shard_{idx:05d}  {len(tok_buf):>6} tokens")

        for d, text in enumerate(docs):
            ids = self.tok.encode(text) + [self.tok.EOS]   # EOS separates docs
            tok_buf.extend(ids)
            doc_buf.extend([d] * len(ids))
            while len(tok_buf) >= self.shard_size:         # emit full shards
                flush(tok_buf[:self.shard_size], doc_buf[:self.shard_size], idx)
                tok_buf = tok_buf[self.shard_size:]
                doc_buf = doc_buf[self.shard_size:]
                idx += 1
        if tok_buf:                                        # final partial shard
            flush(tok_buf, doc_buf, idx)
        return sorted(glob.glob(os.path.join(split_dir, "shard_*.tok.npy")))

    def build(self, corpus):
        docs = list(corpus.docs())
        n_val = max(1, int(len(docs) * self.val_frac))
        val_docs, train_docs = docs[:n_val], docs[n_val:]
        print(f"docs: {len(train_docs)} train / {len(val_docs)} val")
        print("writing train shards:")
        train_paths = self._write_split(train_docs, "train")
        print("writing val shards:")
        val_paths = self._write_split(val_docs, "val")
        print(f"done: {len(train_paths)} train + {len(val_paths)} val shards")
        return train_paths, val_paths


# ---- STAGE 2: load a token budget of shards into RAM -> batches -----------
class ShardStream:
    """Stream (tokens, doc_id) (B, T) batches from a split's shards.

    - Loads only as many shards as needed to reach max_tokens (None = all),
      straight into RAM ONCE. 50M tokens = ~100 MB tokens + ~200 MB doc ids.
      After that, every batch is pure in-memory indexing: no file opens.
    - Non-overlapping blocks: the loaded tokens are cut into back-to-back
      T-length blocks so no token window is ever reused.
    - Rank partition: all ranks share one seed -> identical global block shuffle,
      then rank r takes blocks[r::world_size] -> disjoint data, no overlap.
    - Defaults rank=0/world_size=1 -> single process gets ALL blocks.
    - Yields numpy int64 arrays; the trainer wraps them in torch tensors.
    """

    def __init__(self, split_dir, B, T, seed=0, max_tokens=None, max_shards=None,
                 rank=0, world_size=1):
        paths = sorted(glob.glob(os.path.join(split_dir, "shard_*.tok.npy")))
        if max_shards is not None:                 # None = all shards; int = first N
            paths = paths[:max_shards]
        if not paths:
            raise SystemExit(f"no shards in {split_dir}/")

        toks, docs, have = [], [], 0
        for p in paths:                            # stop once the budget is met
            if max_tokens is not None and have >= max_tokens:
                break
            toks.append(np.load(p))
            docs.append(np.load(p.replace(".tok.npy", ".doc.npy")))
            have += len(toks[-1])
        self.toks = np.concatenate(toks)
        self.docs = np.concatenate(docs)
        if max_tokens is not None:                 # trim to the exact budget
            self.toks = self.toks[:max_tokens]
            self.docs = self.docs[:max_tokens]

        self.B, self.T = B, T
        self.rank, self.world_size = rank, world_size
        self.rng = np.random.default_rng(seed)     # SHARED across ranks (same seed)
        self.n_blocks = len(self.toks) // T        # global block id runs 0..n_blocks-1
        self.offsets = np.arange(T)
        print(f"{split_dir}: {len(toks)} shards, {len(self.toks):,} tokens, "
              f"{self.n_blocks:,} blocks", flush=True)

    def _batch(self, block_ids):
        idx = block_ids[:, None] * self.T + self.offsets   # (B, T) positions
        return self.toks[idx].astype(np.int64), self.docs[idx].astype(np.int64)

    def epoch(self):
        order = self.rng.permutation(self.n_blocks)      # same on all ranks (shared seed)
        mine = order[self.rank::self.world_size]          # disjoint stride partition
        for i in range(0, len(mine) - self.B + 1, self.B):
            yield self._batch(mine[i:i + self.B])         # one vectorized gather

    def iter_batches(self):
        """Infinite stream: loop epochs forever (for a fixed step budget)."""
        while True:
            yield from self.epoch()


def preprocess_to_shards(cfg, raw_dir="data/raw", shard_dir="data/shards",
                         shard_size=1000, val_frac=0.1, overwrite=False):
    """One-time preprocess: read the raw folder and write token shards.

    Skips if shards already exist (pass overwrite=True to force a rebuild).
    """
    have = glob.glob(os.path.join(shard_dir, "train", "shard_*.tok.npy"))
    if have and not overwrite:
        print(f"shards already in {shard_dir}/ -- skip (overwrite=True to rebuild)")
        return
    corpus = RawCorpus(raw_dir)
    if len(corpus) == 0:
        raise SystemExit(f"no .txt files in {raw_dir}/ -- drop documents there first")
    print(f"loaded {len(corpus)} docs from {raw_dir}/")
    ShardWriter(cfg.TOK, out_dir=shard_dir, shard_size=shard_size,
                val_frac=val_frac).build(corpus)


def preprocess_split(cfg, raw_dir, split, shard_dir="data/shards", shard_size=1_000_000):
    """Tokenize ALL docs in raw_dir into shards under shard_dir/<split>/.

    Unlike preprocess_to_shards (which splits ONE folder by val_frac), this writes
    a given folder to a named split -- use it to build train and val from the two
    separate TinyStories files (no leakage, real held-out val).
    """
    corpus = RawCorpus(raw_dir)
    if len(corpus) == 0:
        raise SystemExit(f"no .txt files in {raw_dir}/")
    writer = ShardWriter(cfg.TOK, out_dir=shard_dir, shard_size=shard_size)
    paths = writer._write_split(list(corpus.docs()), split)
    print(f"{split}: {len(paths)} shards from {raw_dir}/")
    return paths


def get_streams(cfg, shard_dir="data/shards", micro_B=4, seed=0,
                train_tokens=50_000_000, val_tokens=None, max_shards=None,
                rank=0, world_size=1):
    """Load token budgets into RAM; return (train_stream, val_stream).

    train_tokens: tokens to load for training (default 50M). None = all shards.
    val_tokens:   tokens to load for validation. None = all val shards.
    max_shards:   optional extra cap on shard count (e.g. 1 for quick debugging).
    rank/world_size: pass the SAME seed on every rank; each rank gets a disjoint
                     slice of blocks. Defaults (0, 1) = single process, all data.
    train_stream: infinite micro-batch iterator for the training loop.
    val_stream:   a callable returning one finite pass over val.
    """
    train = ShardStream(os.path.join(shard_dir, "train"), B=micro_B, T=cfg.T,
                        seed=seed, max_tokens=train_tokens, max_shards=max_shards,
                        rank=rank, world_size=world_size)
    val = ShardStream(os.path.join(shard_dir, "val"), B=micro_B, T=cfg.T,
                      seed=seed + 123, max_tokens=val_tokens, max_shards=max_shards,
                      rank=rank, world_size=world_size)
    return train.iter_batches(), val.epoch


if __name__ == "__main__":
    # Run once (offline): train BPE on data/raw, then raw docs -> token shards.
    # overwrite=True so old (gpt2-tokenized) shards are rebuilt with the new vocab.
    cfg = Config()
    print(f"vocab V = {cfg.V}  (BPE {cfg.V - 2} + PAD + EOS)")
    preprocess_to_shards(cfg, overwrite=True)

Overwriting data_pipeline.py


In [14]:
%%writefile learn_pipeline.py
"""
learn_pipeline.py -- PIPELINE B: token shards -> batches -> probabilities.

All DATA handling (tokenizer, shard writing, shard streaming) lives in
data_pipeline.py. This file starts from the streamed batches and turns them
into logits -> probabilities -> loss (the learning half).

    python data_pipeline.py     # once: raw -> token shards
    python learn_pipeline.py    # every run: shards -> batches (-> training)
"""

import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from data_pipeline import Config, get_streams, Tokenizer

# build the tokenizer ONCE (module-level); reloading it per make_masks call was
# reading the BPE files off disk every step -> a real per-step slowdown
_TOK = None

def make_masks(tokens,doc_id):
    global _TOK
    if _TOK is None:
        _TOK = Tokenizer()          # first call only; cached thereafter
    TOK = _TOK
    B,T = tokens.shape
    # no peeking into future tokens
    casual = torch.tril(torch.ones(T,T, dtype=torch.bool, device=tokens.device)) # (T,T)
    # only attend to tokens within the same document
    same = doc_id[:,:,None] == doc_id[:,None,:] # (B,T,1) vs (B,1,T) -> (B,T,T)
    # combine causal and same-document masks
    attn = (casual & same).unsqueeze(1) # (B,1,T,T)
    # target tokens (shifted by one)
    tgt = tokens[:,1:]
    # identify positions where the document changes
    join = doc_id[:,1:] != doc_id[:,:-1] # (B,T-1) vs (B,T-1) -> comparing adjacent doc_ids
    # positions where the document changes (to avoid computing loss across document boundaries)
    loss = (tgt != TOK.PAD) & (~join)
    return attn, loss

class RMSNorm(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.d = d
        self.g = nn.Parameter(torch.ones(d))
    def forward(self,x):
        # RMSNorm forward pass
        ms = x.pow(2).mean(-1, keepdim=True) # (B,T,D) -> (B,T,1)
        return x*torch.rsqrt(ms + 1e-6)*self.g

class Attention(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.c = c
        self.q = nn.Linear(c.D, c.D, bias=False)
        self.k = nn.Linear(c.D, c.D, bias=False)
        self.v = nn.Linear(c.D, c.D, bias=False)
        self.out = nn.Linear(c.D, c.D, bias=False)
    def forward(self,x,mask):
        c  = self.c
        B,T,_ = x.shape
        sp = lambda t: t.view(B,T,c.H, c.HD).transpose(1,2)
        # get q,k,v projections for multi-head attention
        q,k,v = sp(self.q(x)), sp(self.k(x)), sp(self.v(x)) # (B,c.H,T,c.HD)
        # multiply queries and keys to get attention scores
        s = (q @ k.transpose(-2,-1)) / (c.HD ** 0.5) # (B,c.H,T,c.HD) @ (B,c.H,c.HD,T) -> (B,c.H,T,T)
        # apply the attention mask to prevent attending to future tokens or tokens from different documents
        s = s.masked_fill(~mask, float('-inf'))
        # softmax to obtain attention weights
        w = torch.nan_to_num(s.softmax(-1))
        # value vectors are combined with attention weights to produce the context vectors
        ctx = (w @ v).transpose(1,2).reshape(B,T,c.D) 
        return self.out(ctx)

class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.D, c.DFF, bias=False)
        self.up = nn.Linear(c.D, c.DFF, bias=False)
        self.bottom = nn.Linear(c.DFF, c.D, bias=False)
    def forward(self,x):
        return self.bottom(self.up(x) * F.silu(self.gate(x)))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.attn = Attention(c)
        self.n1 = RMSNorm(c.D)
        self.n2 = RMSNorm(c.D)
        self.ffn = SwiGLU(c)
    def forward(self,x,mask):
        x = x + self.attn(self.n1(x), mask)
        x = x + self.ffn(self.n2(x))
        return x


class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok_emb = nn.Embedding(c.V, c.D)
        self.pos_emb = nn.Embedding(c.T, c.D)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.L)])
        self.ns = RMSNorm(c.D)
        self.head = nn.Linear(c.D, c.V, bias=False)

    def forward(self,tokens, mask):
        B,T = tokens.shape
        pos = torch.arange(T, device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x,mask)
        x = self.ns(x)
        logits = self.head(x)
        return logits

def loss_fn(logits, tokens, loss_mask):
    tgt = tokens[:, 1:].reshape(-1) # (B, T-1) -> (B*(T-1))
    lg = logits[:,:-1].reshape(-1,logits.size(-1)) # (B, T-1, V) -> (B*(T-1), V)
    nll = F.cross_entropy(lg, tgt, reduction='none')
    m = loss_mask.reshape(-1).float()
    loss = (nll * m).sum() / m.sum().clamp(min=1)
    return loss, int(m.sum())

def lr_at(s, WARMUP, STEPS):
    if s < WARMUP:
        return s / WARMUP
    t = (s - WARMUP) / max(1, STEPS - WARMUP)
    return 0.1 + 0.45 * (1 + math.cos(math.pi * t))


# @torch.no_grad()
# def evaluate(model, val_stream, max_batches=20):
#     """Token-weighted mean loss over up to max_batches of the val stream."""
#     model.eval()
#     tot, ntok = 0.0, 0
#     for i, (x, d) in enumerate(val_stream()):        # fresh finite pass
#         if i >= max_batches:
#             break
#         tokens, doc_id = torch.from_numpy(x), torch.from_numpy(d)
#         attn, loss_mask = make_masks(tokens, doc_id)
#         loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
#         tot += loss.item() * n
#         ntok += n
#     model.train()
#     return tot / max(1, ntok)


# def _trace_shapes(model, tokens, doc_id):
#     """One-time forward pass that prints the output shape of every submodule."""
#     attn, loss_mask = make_masks(tokens, doc_id)
#     print("  inputs:")
#     print(f"    {'tokens':<26} {tuple(tokens.shape)}")
#     print(f"    {'doc_id':<26} {tuple(doc_id.shape)}")
#     print(f"    {'attn mask':<26} {tuple(attn.shape)}")
#     print(f"    {'loss mask':<26} {tuple(loss_mask.shape)}")
#     print("  module outputs:")

#     handles = []
#     def mk(name):
#         def hook(mod, inp, out):
#             if isinstance(out, torch.Tensor):
#                 print(f"    {name:<26} {tuple(out.shape)}")
#         return hook
#     for name, mod in model.named_modules():
#         if name:                                     # skip the root module
#             handles.append(mod.register_forward_hook(mk(name)))
#     with torch.no_grad():
#         logits = model(tokens, attn)
#     for h in handles:
#         h.remove()
#     print(f"    {'logits (head)':<26} {tuple(logits.shape)}")


# def train(STEPS=500, WARMUP=50, lr=3e-4, micro_B=4, accum_steps=4,
#           eval_every=100, log_every=1, max_shards=None, seed=0, debug=True):
#     cfg = Config()
#     torch.manual_seed(seed)

#     # STAGE 2: stream (tokens, doc_id) batches from the saved token shards.
#     train_stream, val_stream = get_streams(cfg, micro_B=micro_B, seed=seed,
#                                            max_shards=max_shards)
#     model = GPT(cfg)
#     opt = torch.optim.AdamW(model.parameters(), lr=lr)
#     print(f"eff. batch {micro_B * accum_steps}  |  ln(V) {math.log(cfg.V):.3f}")

#     if debug:                                        # one-time shape trace
#         x, d = next(train_stream)
#         print("shape trace:")
#         _trace_shapes(model, torch.from_numpy(x), torch.from_numpy(d))

#     seen = 0                                         # cumulative tokens trained on
#     for s in range(STEPS):
#         t0 = time.time()
#         cur_lr = lr * lr_at(s, WARMUP, STEPS)
#         for g in opt.param_groups:                   # LR warmup + cosine decay
#             g['lr'] = cur_lr
#         opt.zero_grad(set_to_none=True)

#         run_loss, step_tok = 0.0, 0
#         for _ in range(accum_steps):                 # accumulate micro-batches
#             x, d = next(train_stream)
#             tokens, doc_id = torch.from_numpy(x), torch.from_numpy(d)
#             attn, loss_mask = make_masks(tokens, doc_id)
#             loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
#             (loss / accum_steps).backward()          # grads add up; no step yet
#             run_loss, step_tok = run_loss + loss.item() * n, step_tok + n
#         gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         opt.step()                                   # one update per effective batch
#         seen += step_tok

#         if s % log_every == 0:
#             dt = time.time() - t0
#             avg = run_loss / max(1, step_tok)        # token-weighted mean over the step
#             print(f"step {s:4d} | lr {cur_lr:.2e} | loss {avg:.4f} | "
#                   f"gnorm {float(gnorm):.2f} | tok {step_tok} | seen {seen} | "
#                   f"{step_tok / max(dt, 1e-9):.0f} tok/s")
#         if s % eval_every == 0:
#             print(f"           val {evaluate(model, val_stream):.4f}")
#     return model


if __name__ == "__main__":
    # Requires token shards on disk first:  python data_pipeline.py
    # train(STEPS=500, eval_every=50)
    pass


Overwriting learn_pipeline.py


In [15]:
%%writefile reversible_pipeline.py

"""
reversible_pipeline.py -- reversible transformer (memory-cheap depth).

Idea: treat each block as an ODE step on TWO coupled streams (x1, x2). Because
the update is an invertible coupling, the backward pass can RECONSTRUCT every
block's input from its output instead of storing activations. Peak activation
memory becomes ~ONE block instead of L blocks -> you can push a bigger batch.

Reuses Attention / SwiGLU / RMSNorm from learn_pipeline so the parameter count
stays identical to the baseline GPT (same ~20M model, just trained reversibly).

Variants (which integrator couples the two streams):
    euler     additive coupling, 2 fn evals/block (F, G)   -- fastest
    midpoint  symmetric Verlet coupling, 3 fn evals (G,F,G) -- 2nd order, stabler

Both are EXACTLY reversible (reconstruction is algebraic, not a solve).
"""

import torch
import torch.nn as nn

from data_pipeline import Config
from learn_pipeline import Attention, SwiGLU, RMSNorm


class ReversibleBlock(nn.Module):
    """One invertible block over two streams.

    F = RMSNorm -> Attention   (mixes across time)
    G = RMSNorm -> SwiGLU      (mixes across channels)
    """

    def __init__(self, c, variant="euler"):
        super().__init__()
        self.variant = variant
        self.n1 = RMSNorm(c.D)
        self.attn = Attention(c)
        self.n2 = RMSNorm(c.D)
        self.ffn = SwiGLU(c)

    def F(self, x, mask):
        return self.attn(self.n1(x), mask)

    def G(self, x):
        return self.ffn(self.n2(x))

    def forward(self, x1, x2, mask):
        if self.variant == "euler":
            y1 = x1 + self.F(x2, mask)
            y2 = x2 + self.G(y1)
        else:  # midpoint (symmetric Verlet): half G, full F, half G
            h = x2 + 0.5 * self.G(x1)
            y1 = x1 + self.F(h, mask)
            y2 = h + 0.5 * self.G(y1)
        return y1, y2

    def inverse(self, y1, y2, mask):
        """Exactly undo forward(): reconstruct (x1, x2) from (y1, y2)."""
        if self.variant == "euler":
            x2 = y2 - self.G(y1)
            x1 = y1 - self.F(x2, mask)
        else:  # midpoint
            h = y2 - 0.5 * self.G(y1)
            x1 = y1 - self.F(h, mask)
            x2 = h - 0.5 * self.G(x1)
        return x1, x2

    def backward_pass(self, y1, y2, dy1, dy2, mask):
        """Reconstruct this block's input, recompute its forward WITH grad, and
        push (dy1, dy2) back into input grads + accumulate parameter grads."""
        with torch.no_grad():
            x1, x2 = self.inverse(y1, y2, mask)
        x1 = x1.detach().requires_grad_(True)
        x2 = x2.detach().requires_grad_(True)
        with torch.enable_grad():
            z1, z2 = self.forward(x1, x2, mask)
            torch.autograd.backward((z1, z2), (dy1, dy2))
        return x1.detach(), x2.detach(), x1.grad, x2.grad


class _ReversibleFunction(torch.autograd.Function):
    """Runs the whole stack with NO stored activations; rebuilds them in backward."""

    @staticmethod
    def forward(ctx, x, mask, blocks):
        with torch.no_grad():
            x1 = x2 = x
            for blk in blocks:
                x1, x2 = blk.forward(x1, x2, mask)
        ctx.blocks = blocks
        ctx.mask = mask
        ctx.save_for_backward(x1, x2)          # only the FINAL states, not per-layer
        return x1 + x2

    @staticmethod
    def backward(ctx, dout):
        y1, y2 = ctx.saved_tensors
        dy1 = dy2 = dout                        # output was y1 + y2
        for blk in reversed(ctx.blocks):
            y1, y2, dy1, dy2 = blk.backward_pass(y1, y2, dy1, dy2, ctx.mask)
        return dy1 + dy2, None, None           # x was fed to both streams


class ReversibleGPT(nn.Module):
    """Same shape/params as learn_pipeline.GPT, but the block stack is reversible."""

    def __init__(self, c, variant="euler"):
        super().__init__()
        self.c = c
        self.variant = variant
        self.tok_emb = nn.Embedding(c.V, c.D)
        self.pos_emb = nn.Embedding(c.T, c.D)
        self.blocks = nn.ModuleList([ReversibleBlock(c, variant) for _ in range(c.L)])
        self.ns = RMSNorm(c.D)
        self.head = nn.Linear(c.D, c.V, bias=False)

    def forward(self, tokens, mask):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        x = _ReversibleFunction.apply(x, mask, self.blocks)
        x = self.ns(x)
        return self.head(x)


if __name__ == "__main__":
    # quick reversibility self-check: inverse() must undo forward() to ~fp32 eps
    cfg = Config()
    cfg.D, cfg.H, cfg.L, cfg.T = 384, 8, 7, 256
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    for variant in ("euler", "midpoint"):
        blk = ReversibleBlock(cfg, variant)
        x1 = torch.randn(2, cfg.T, cfg.D)
        x2 = torch.randn(2, cfg.T, cfg.D)
        mask = torch.ones(2, 1, cfg.T, cfg.T, dtype=torch.bool).tril(0)
        y1, y2 = blk.forward(x1, x2, mask)
        r1, r2 = blk.inverse(y1, y2, mask)
        err = (r1 - x1).abs().max().item() + (r2 - x2).abs().max().item()
        print(f"{variant:9s} reconstruction error = {err:.2e}")


Writing reversible_pipeline.py


In [16]:
%%writefile reversible_launch.py

"""
reversible_launch.py -- trainer for the 20M model (single-GPU or multi-GPU DDP).

Runs the SAME ~20M-param model three ways so you can fill in the assignment table:

  1) baseline (store activations)     -- fix a batch you can run
  2) reversible, SAME batch           -- isolate the reversibility effect
  3) reversible, MAX batch            -- spend the freed memory on throughput

Single GPU:
    python reversible_launch.py --micro-B 16 --tag baseline

Multi-GPU (data-parallel across all GPUs) -- launch with torchrun:
    torchrun --nproc_per_node=2 reversible_launch.py --reversible --variant euler \
             --micro-B 16 --tag rev_same

NOTE on ZeRO-3: FSDP is NOT used here. It shards params/optimizer (useless for a
tiny 20M model) and its just-in-time param hooks are bypassed by the reversible
custom autograd.Function. We use plain data parallelism instead: each GPU holds a
full replica over a disjoint data slice, and grads are averaged by an explicit
all_reduce (the DDP wrapper's bucketed reduction can miss the reversible grads,
which are produced inside a nested backward).

Each run writes metrics_<tag>.csv and prints a final summary line
(final loss | tok/s | peak memory) that drops straight into README.md.

Needs token shards first:  python data_pipeline.py
(default --shard-dir is the Kaggle dataset shards, so the flag is optional there)
"""

import argparse
import csv
import math
import os
import time

import torch
import torch.distributed as dist

from data_pipeline import Config, get_streams
from learn_pipeline import GPT, make_masks, loss_fn, lr_at
from reversible_pipeline import ReversibleGPT

# Kaggle dataset root holding the token shards (matches zero3_launch.py); the
# default --shard-dir points here so you can omit the flag on Kaggle.
SRC = "/kaggle/input/datasets/gona26/tinystories-data-540m"
DEFAULT_SHARDS = f"{SRC}/shards"


def ensure_bpe(src=f"{SRC}/bpe", dst="data/bpe"):
    """Tokenizer/Config hard-code data/bpe; symlink it to the dataset so the run
    is self-contained (no separate setup cell needed). Idempotent + rank-safe."""
    if os.path.exists(dst) or not os.path.isdir(src):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    try:
        os.symlink(src, dst)
    except FileExistsError:                      # another rank won the race
        pass


def setup_dist():
    """Join the process group if launched by torchrun; else run single-process.

    Returns (rank, world_size, local_rank, distributed?).
    """
    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        dist.init_process_group(backend="nccl",
                                device_id=torch.device(f"cuda:{local_rank}"))
        return dist.get_rank(), dist.get_world_size(), local_rank, True
    return 0, 1, 0, False


def build_config():
    # pin the ~19.6M ("20M") config regardless of data_pipeline defaults
    cfg = Config()
    cfg.D, cfg.H, cfg.L = 384, 8, 7
    cfg.T = 256
    cfg.HD, cfg.DFF = cfg.D // cfg.H, cfg.D * 4
    return cfg


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def model_info(model, cfg):
    P = n_params(model)
    emb = 2 * cfg.V * cfg.D                     # tok_emb + head dominate a tiny model
    return (f"model: {P/1e6:.2f}M params | L={cfg.L} D={cfg.D} H={cfg.H} "
            f"T={cfg.T} V={cfg.V} | embed+head ~{emb/1e6:.1f}M ({100*emb/P:.0f}%)")


def measure_holdings(model, opt):
    # live bytes held per GPU: full replica here (NOT sharded, unlike ZeRO-3)
    mb = lambda ts: sum(t.numel() * t.element_size() for t in ts) / 1e6
    params_mb = mb(list(model.parameters()))
    grads_mb = mb([p.grad for p in model.parameters() if p.grad is not None])
    optim_mb = mb([t for st in opt.state.values() for t in st.values() if torch.is_tensor(t)])
    return params_mb, grads_mb, optim_mb


@torch.no_grad()
def recon_error(model, tokens, mask):
    """End-to-end reversibility drift: run the block stack forward, invert it, and
    measure how far reconstruction misses the input (exact rev -> ~1e-5 in fp32)."""
    pos = torch.arange(tokens.shape[1], device=tokens.device)
    x = model.tok_emb(tokens) + model.pos_emb(pos)
    x1 = x2 = x
    for blk in model.blocks:
        x1, x2 = blk.forward(x1, x2, mask)
    for blk in reversed(model.blocks):
        x1, x2 = blk.inverse(x1, x2, mask)
    return (x1 - x).abs().max().item() + (x2 - x).abs().max().item()


@torch.no_grad()
def evaluate(model, val_stream, device, max_batches, distributed=False):
    """Token-weighted mean val loss over up to max_batches (perplexity = e^loss).

    Under DDP, sums (loss*tok, tok) across ranks -> a true GLOBAL val loss.
    """
    model.eval()
    tot, ntok = 0.0, 0
    for i, (x, d) in enumerate(val_stream()):
        if i >= max_batches:
            break
        tokens = torch.from_numpy(x).to(device)
        doc_id = torch.from_numpy(d).to(device)
        attn, loss_mask = make_masks(tokens, doc_id)
        loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
        tot += loss.item() * n
        ntok += n
    model.train()
    if distributed:
        t = torch.tensor([tot, float(ntok)], device=device)
        dist.all_reduce(t)
        return (t[0] / t[1].clamp(min=1)).item()
    return tot / max(1, ntok)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--reversible", action="store_true",
                    help="use the reversible stack (else the baseline GPT)")
    ap.add_argument("--variant", choices=["euler", "midpoint"], default="euler")
    ap.add_argument("--micro-B", type=int, default=16, help="batch (sequences) per step")
    ap.add_argument("--accum", type=int, default=1, help="gradient accumulation steps")
    ap.add_argument("--train-tokens", type=int, default=50_000_000)
    ap.add_argument("--shard-dir", default=DEFAULT_SHARDS,
                    help="token-shard folder (defaults to the Kaggle dataset shards)")
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--warmup", type=int, default=300)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--eval-every", type=int, default=500)
    ap.add_argument("--eval-batches", type=int, default=50)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--tag", default="run")
    args = ap.parse_args()

    rank, world_size, local_rank, distributed = setup_dist()
    is0 = rank == 0
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{local_rank}")
    else:
        device = torch.device("cpu")
    torch.manual_seed(args.seed)      # SAME on every rank -> identical model replicas

    ensure_bpe()                      # link data/bpe -> dataset before Tokenizer loads
    cfg = build_config()
    # each rank gets a DISJOINT slice of blocks (shared seed + rank stride)
    train_stream, val_stream = get_streams(
        cfg, shard_dir=args.shard_dir, micro_B=args.micro_B, seed=1234,
        train_tokens=args.train_tokens, val_tokens=None,
        rank=rank, world_size=world_size)

    # GLOBAL tokens per optimizer step (all ranks) -> steps to cover the budget
    tok_per_step = cfg.T * args.micro_B * args.accum * world_size
    STEPS = math.ceil(args.train_tokens / tok_per_step)

    if args.reversible:
        model = ReversibleGPT(cfg, variant=args.variant).to(device)
        kind = f"reversible/{args.variant}"
        fn_evals = 2 if args.variant == "euler" else 3   # F,G vs G,F,G per block
    else:
        model = GPT(cfg).to(device)
        kind = "baseline"
        fn_evals = 2
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)

    if is0:
        print(model_info(model, cfg), flush=True)
        print(f"{kind} | world_size={world_size} micro_B={args.micro_B} "
              f"accum={args.accum} -> global batch "
              f"{args.micro_B * args.accum * world_size} | {fn_evals} fn-evals/block "
              f"| STEPS={STEPS} | budget {args.train_tokens/1e6:.0f}M tok | {device}",
              flush=True)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    if is0:
        logf = open(f"metrics_{args.tag}.csv", "w", newline="")
        writer = csv.writer(logf)
        writer.writerow(["step", "tokens", "loss_tokens", "loss", "ppl", "val_loss",
                         "val_ppl", "grad_norm", "tok_per_s", "lr", "alloc_mb",
                         "peak_mb", "reserved_mb", "gpu_used_mb", "gpu_total_mb",
                         "recon_err"])

    # token counters from ACTUAL batch shapes; window counters -> steady tok/s
    local_seen = 0
    local_trained = 0.0
    win_tok, win_t0 = 0, time.time()
    tps_hist = []
    final_loss = float("nan")
    last_recon = ""
    t_train = time.time()
    for s in range(1, STEPS + 1):
        cur_lr = args.lr * lr_at(s, args.warmup, STEPS)
        for g in opt.param_groups:
            g["lr"] = cur_lr

        opt.zero_grad(set_to_none=True)
        run_loss, step_tok = 0.0, 0
        for _ in range(args.accum):
            x, d = next(train_stream)
            local_seen += x.shape[0] * x.shape[1]
            win_tok += x.shape[0] * x.shape[1]
            tokens = torch.from_numpy(x).to(device)
            doc_id = torch.from_numpy(d).to(device)
            attn, loss_mask = make_masks(tokens, doc_id)
            loss, n = loss_fn(model(tokens, attn), tokens, loss_mask)
            (loss / args.accum).backward()
            run_loss += loss.item() * n
            step_tok += n
            local_trained += n

        if distributed:
            # manual DDP: average grads across ranks (the reversible custom
            # autograd.Function bypasses DDP's bucketed hooks, so reduce by hand)
            for p in model.parameters():
                if p.grad is not None:
                    dist.all_reduce(p.grad)
                    p.grad /= world_size
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if s % args.log_every == 0 or s == STEPS:
            # one reduce for: sum loss*tok, sum tok, global seen, global trained, window tok
            red = torch.tensor([run_loss, float(step_tok), float(local_seen),
                                float(local_trained), float(win_tok)],
                               device=device, dtype=torch.float64)
            if distributed:
                dist.all_reduce(red)
            avg_loss = (red[0] / red[1].clamp(min=1)).item()
            seen = int(red[2].item())
            trained = int(red[3].item())
            final_loss = avg_loss
            ppl = math.exp(min(avg_loss, 20))

            win_dt = time.time() - win_t0
            tps = red[4].item() / max(win_dt, 1e-9)       # steady window throughput
            tps_hist.append(tps)
            eta_min = (STEPS - s) * (win_dt / args.log_every) / 60

            # 4 memory levels (each contains the one before), + GPU total; MAX over ranks
            if device.type == "cuda":
                free_b, total_b = torch.cuda.mem_get_info()
                mem = torch.tensor([torch.cuda.memory_allocated(),
                                    torch.cuda.max_memory_allocated(),
                                    torch.cuda.memory_reserved(),
                                    total_b - free_b, total_b],
                                   device=device, dtype=torch.float64) / 1e6
                if distributed:
                    dist.all_reduce(mem, op=dist.ReduceOp.MAX)
                alloc_mb, peak_mb, resv_mb, used_mb, total_mb = mem.tolist()
            else:
                alloc_mb = peak_mb = resv_mb = used_mb = total_mb = 0.0

            val_loss = val_ppl = ""
            if s % args.eval_every == 0 or s == STEPS:
                vl = evaluate(model, val_stream, device, args.eval_batches, distributed)
                val_loss, val_ppl = f"{vl:.4f}", f"{math.exp(min(vl, 20)):.2f}"

            # reversibility-specific: confirm inverse() still reconstructs the input
            recon = ""
            if args.reversible and (s % args.eval_every == 0 or s == STEPS):
                recon = f"{recon_error(model, tokens, attn):.2e}"
                last_recon = recon

            if is0:
                writer.writerow([s, seen, trained, f"{avg_loss:.4f}", f"{ppl:.2f}",
                                 val_loss, val_ppl, f"{float(gnorm):.2f}",
                                 f"{tps:.0f}", f"{cur_lr:.2e}",
                                 f"{alloc_mb:.0f}", f"{peak_mb:.0f}", f"{resv_mb:.0f}",
                                 f"{used_mb:.0f}", f"{total_mb:.0f}", recon])
                logf.flush()
                msg = (f"step {s:5d}/{STEPS} | {seen/1e6:5.1f}M tok "
                       f"({trained/1e6:.1f}M in loss) | loss {avg_loss:.4f} ppl {ppl:.1f}")
                if val_ppl:
                    msg += f" | val_ppl {val_ppl}"
                msg += (f" | gnorm {float(gnorm):.2f} | {tps:.0f} tok/s | "
                        f"mem peak {peak_mb:.0f} / resv {resv_mb:.0f} / "
                        f"gpu {used_mb:.0f} of {total_mb:.0f}MB | eta {eta_min:.0f}m")
                if recon:
                    msg += f" | recon {recon}"
                print(msg, flush=True)

            win_tok, win_t0 = 0, time.time()     # reset window after logging

    # final token audit: counted positions vs the budget math
    total_seen = torch.tensor([float(local_seen)], device=device, dtype=torch.float64)
    if distributed:
        dist.all_reduce(total_seen)
    expected = STEPS * cfg.T * args.micro_B * args.accum * world_size

    total_s = time.time() - t_train
    avg_tps = (sum(tps_hist) / len(tps_hist)) if tps_hist else 0.0
    peak_mb = (torch.cuda.max_memory_allocated() / 1e6
               if device.type == "cuda" else 0.0)
    if distributed:
        mp = torch.tensor([peak_mb], device=device)
        dist.all_reduce(mp, op=dist.ReduceOp.MAX)
        peak_mb = mp.item()

    # per-GPU holdings (full replica here -- contrast with ZeRO-3's 1/N shards)
    params_mb, grads_mb, optim_mb = measure_holdings(model, opt)

    if is0:
        logf.close()
        with open(f"holdings_{args.tag}.csv", "w", newline="") as hf:
            w = csv.writer(hf)
            w.writerow(["component", "mb"])
            w.writerow(["params", f"{params_mb:.2f}"])
            w.writerow(["grads", f"{grads_mb:.2f}"])
            w.writerow(["optimizer", f"{optim_mb:.2f}"])
        print(f"\ntoken audit: counted {int(total_seen.item()):,} | "
              f"expected {expected:,} | budget {args.train_tokens:,}", flush=True)
        print("==== SUMMARY (" + kind + ") ====", flush=True)
        print(f"tag           : {args.tag}", flush=True)
        print(f"final loss    : {final_loss:.4f}  (ppl {math.exp(min(final_loss,20)):.2f})",
              flush=True)
        print(f"speed         : {avg_tps:.0f} tok/s (avg, global)", flush=True)
        print(f"peak memory   : {peak_mb:.0f} MB (per GPU)", flush=True)
        print(f"holdings      : params {params_mb:.0f} + grads {grads_mb:.0f} + "
              f"optim {optim_mb:.0f} = {params_mb+grads_mb+optim_mb:.0f} MB "
              f"(per GPU, full replica)", flush=True)
        if args.reversible:
            print(f"recon error   : {last_recon or 'n/a'}  (exact-reversibility drift)",
                  flush=True)
            print(f"fn-evals/blk  : {fn_evals}  (variant {args.variant})", flush=True)
        print(f"wall time     : {total_s/60:.1f} min  ({STEPS} steps)", flush=True)
        print(f"batch         : micro_B={args.micro_B} accum={args.accum} "
              f"world_size={world_size} global={args.micro_B*args.accum*world_size}",
              flush=True)

    if distributed:
        dist.barrier()
        dist.destroy_process_group()


if __name__ == "__main__":
    main()


Writing reversible_launch.py


In [17]:
!torchrun --nproc_per_node=2 reversible_launch.py --reversible --variant euler --micro-B 128 --tag rev_euler

W1002 16:21:38.657000 178 torch/distributed/run.py:852] 
W1002 16:21:38.657000 178 torch/distributed/run.py:852] *****************************************
W1002 16:21:38.657000 178 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 16:21:38.657000 178 torch/distributed/run.py:852] *****************************************
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 50 shards, 50,000,000 tokens, 195,312 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 50 shards, 50,000,000 tokens, 195,312 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
model: 19.69M params | L=7 D=384 H=8 T=256 V=40

In [18]:
!torchrun --nproc_per_node=2 reversible_launch.py --reversible --variant euler --micro-B 512 --train-tokens 55000000 --warmup 20 --tag rev_euler_512b

W1002 16:59:34.951000 211 torch/distributed/run.py:852] 
W1002 16:59:34.951000 211 torch/distributed/run.py:852] *****************************************
W1002 16:59:34.951000 211 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 16:59:34.951000 211 torch/distributed/run.py:852] *****************************************
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
model: 19.69M params | L=7 D=384 H=8 T=256 V=40

In [19]:
!torchrun --nproc_per_node=2 reversible_launch.py --reversible --variant euler --micro-B 768 --train-tokens 55000000 --warmup 20 --tag rev_euler_768b

W1002 17:41:14.450000 244 torch/distributed/run.py:852] 
W1002 17:41:14.450000 244 torch/distributed/run.py:852] *****************************************
W1002 17:41:14.450000 244 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1002 17:41:14.450000 244 torch/distributed/run.py:852] *****************************************
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/train: 55 shards, 55,000,000 tokens, 214,843 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
/kaggle/input/datasets/gona26/tinystories-data-540m/shards/val: 6 shards, 5,616,737 tokens, 21,940 blocks
model: 19.69M params | L=7 D=384 H=8 T=256 V=40

In [20]:
1+1

2